# BCAF / BreaKHis — full magnification and ablation sweep

This notebook completes the experiments the reviewer report lists as CRITICAL and HIGH.
It is designed to survive disconnection: **every finished run is written to disk immediately
and skipped on the next start**, so you can stop and restart as often as you like and it will
pick up exactly where it left off.

**What it produces**

| Section | Output | Review item |
|---|---|---|
| 5 | Concatenation-fusion ablation + Swin-Tiny alone at 40x | CRITICAL |
| 5 | Full 5-model family at 40x, 100x, 200x, 400x | HIGH |
| 6 | Image-wise leakage arm at every magnification | HIGH |
| 7 | Calibration for every model x magnification | — |
| 8 | Explanation faithfulness across **all five folds** | MEDIUM |
| 9 | Statistics across the model family, Holm-corrected | — |

**Before you start**

1. Runtime -> Change runtime type -> **GPU** (A100 or L4 if available).
2. Run the cells in order. Section 2 mounts Google Drive so results persist.
3. Budget roughly **10-13 GPU-hours** for the whole sweep. The run plan is ordered so the
   CRITICAL results arrive in the first hour.
4. If the session drops, just re-run the notebook from the top. Finished runs are skipped.

## 1. Credentials

In [ ]:
# Paste here ONLY if automatic authentication fails. Clear before sharing.
KAGGLE_USERNAME = ""      # e.g. "drqaimmehdi"
KAGGLE_KEY      = ""      # the "key" field from kaggle.json

# Optional. Pretrained ImageNet weights are fetched from Hugging Face, which
# rate-limits anonymous downloads. Without a token you will see:
#   "Please set a HF_TOKEN to enable higher rate limits and faster downloads."
# That is a warning, not an error -- the download still works. A free read-only
# token removes it and makes weight fetching faster and more reliable.
# Get one at huggingface.co -> Settings -> Access Tokens -> New token (read).
HF_TOKEN = ""

import os, json as _json, pathlib as _pl

def configure_kaggle_auth(username="", key="", verbose=True):
    """Set up Kaggle credentials from the first source that supplies them.

    Order of preference:
      1. credentials already present in the environment or ~/.kaggle/kaggle.json
      2. Kaggle Secrets (Add-ons -> Secrets), which keeps the key out of the file
      3. the values pasted above

    Returns the source used, or None if no credentials were found. Finding none
    is not an error inside Kaggle, where the runtime is already authenticated.
    """
    cfg = _pl.Path.home() / ".kaggle" / "kaggle.json"
    if cfg.exists() or (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")):
        if verbose:
            print("Using credentials already present in this environment.")
        return "existing"

    try:                                   # Kaggle Secrets
        from kaggle_secrets import UserSecretsClient
        s = UserSecretsClient()
        u, k = s.get_secret("KAGGLE_USERNAME"), s.get_secret("KAGGLE_KEY")
        if u and k:
            username, key = u, k
            if verbose:
                print("Using credentials from Kaggle Secrets.")
    except Exception:
        pass

    if not (username and key):
        if verbose:
            print("No explicit credentials supplied.")
            print("That is normal inside Kaggle - the runtime is already signed in.")
        return None

    cfg.parent.mkdir(parents=True, exist_ok=True)
    cfg.write_text(_json.dumps({"username": username, "key": key}))
    cfg.chmod(0o600)
    os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = username, key
    if verbose:
        print(f"Credentials configured for user '{username}'.")
    return "supplied"

AUTH_SOURCE = configure_kaggle_auth(KAGGLE_USERNAME, KAGGLE_KEY)


def configure_hf_token(token: str = "", verbose: bool = True):
    """Set a Hugging Face token if one is available, from any source.

    Only affects download rate limits for pretrained weights. Everything works
    without it; this just removes the warning and speeds up the fetch.
    """
    if os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN"):
        if verbose:
            print("Hugging Face token already present in the environment.")
        return True

    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF_TOKEN") or ""
        except Exception:
            token = ""

    if not token:
        if verbose:
            print("No Hugging Face token set. Downloads will still work, but you "
                  "will see a rate-limit warning. This is harmless.")
        return False

    os.environ["HF_TOKEN"] = token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = token
    if verbose:
        print("Hugging Face token configured.")
    return True

configure_hf_token(HF_TOKEN)

No explicit credentials supplied.
That is normal inside Kaggle - the runtime is already signed in.
No Hugging Face token set. Downloads will still work, but you will see a rate-limit warning. This is harmless.


False

## 2. Environment, Drive and output paths

In [ ]:
import os, sys, json, shutil, pathlib, subprocess

IN_COLAB  = "google.colab" in sys.modules or os.path.exists("/content")
IN_KAGGLE = os.path.exists("/kaggle/working")

# The source modules always live here on both platforms; only results move.
SRC = "/kaggle/working/src"
os.makedirs(SRC, exist_ok=True)
sys.path.insert(0, SRC)

OUT = None
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        OUT = "/content/drive/MyDrive/bcaf_sweep"
        print("Results will persist in Google Drive -> MyDrive/bcaf_sweep")
    except Exception as e:
        print("Drive not mounted (%s). Falling back to local storage." % e)
        print("WARNING: local storage is lost when the session ends.")
        OUT = "/content/bcaf_sweep"
elif IN_KAGGLE:
    OUT = "/kaggle/working"
else:
    OUT = os.path.abspath("./bcaf_sweep")

CKPT = f"{OUT}/checkpoints"
os.makedirs(OUT, exist_ok=True)
os.makedirs(CKPT, exist_ok=True)
print("OUT  =", OUT)
print("CKPT =", CKPT)

Mounted at /content/drive
Results will persist in Google Drive -> MyDrive/bcaf_sweep
OUT  = /content/drive/MyDrive/bcaf_sweep
CKPT = /content/drive/MyDrive/bcaf_sweep/checkpoints


## 3. Project source

In [ ]:
%%writefile /kaggle/working/src/data.py
"""
BreakHis data layer with strict patient-wise splitting.

The central correctness requirement of this project: no patient may ever appear
in both a training and an evaluation fold. Every split produced here is checked
by an assertion that fails loudly rather than silently degrading into leakage.

Filename grammar (BreaKHis_v1):
    SOB_B_A-14-22549AB-40-001.png
    |   | |  |          |   `-- sequence number
    |   | |  |          `------ magnification (40|100|200|400)
    |   | |  `----------------- slide / patient identifier (14-22549AB)
    |   | `-------------------- subtype code (A,F,PT,TA | DC,LC,MC,PC)
    |   `---------------------- class (B=benign, M=malignant)
    `-------------------------- biopsy procedure (SOB)
"""
from __future__ import annotations

import re
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

# ----------------------------------------------------------------------------
# Label vocabulary
# ----------------------------------------------------------------------------
BINARY_CLASSES = ("benign", "malignant")

SUBTYPE_NAMES = {
    "A": "adenosis",
    "F": "fibroadenoma",
    "PT": "phyllodes_tumor",
    "TA": "tubular_adenoma",
    "DC": "ductal_carcinoma",
    "LC": "lobular_carcinoma",
    "MC": "mucinous_carcinoma",
    "PC": "papillary_carcinoma",
}
SUBTYPE_ORDER = ("A", "F", "PT", "TA", "DC", "LC", "MC", "PC")
SUBTYPE_TO_IDX = {code: i for i, code in enumerate(SUBTYPE_ORDER)}

MAGNIFICATIONS = (40, 100, 200, 400)

#: Extensions treated as candidate images when locating the dataset.
IMAGE_EXTS = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")

# SOB_B_A-14-22549AB-40-001.png
_FNAME_RE = re.compile(
    r"^SOB[-_](?P<cls>[BM])[-_](?P<subtype>[A-Z]{1,2})[-_]"
    r"(?P<patient>\d{2}-\d+[A-Z]*)[-_]"
    r"(?P<mag>40|100|200|400)[-_]"
    r"(?P<seq>\d+)\.(?:png|jpg|jpeg|tif|tiff|bmp)$",
    re.IGNORECASE,
)


#: Slide/patient identifiers look like "14-22549AB" wherever they appear.
_PATIENT_RE = re.compile(r"(\d{2}-\d+[A-Z]*)", re.IGNORECASE)


def infer_from_path(path: Path, root: Path) -> dict | None:
    """Recover labels from folder structure when filenames have been renamed.

    Some BreakHis mirrors flatten or rebuild the directory tree and rename the
    files, which defeats the filename grammar. Where the folders still encode
    class, subtype, magnification and slide id, we can recover the record.

    Returns None if the patient identifier cannot be recovered. That case must
    NOT be silently tolerated: without patient ids there is no patient-wise
    split, and the central claim of this study cannot be evaluated.
    """
    try:
        parts = path.relative_to(root).parts
    except ValueError:
        parts = path.parts
    lowered = [x.lower() for x in parts]

    if any("malignant" in x for x in lowered):
        binary = 1
    elif any("benign" in x for x in lowered):
        binary = 0
    else:
        binary = None

    code = None
    for x in parts:
        xl = x.lower()
        for c, name in SUBTYPE_NAMES.items():
            if xl == name or xl == c.lower():
                code = c
                break
        if code:
            break

    mag = None
    for x in parts:
        m = re.fullmatch(r"(40|100|200|400)\s*x?", x, re.IGNORECASE)
        if m:
            mag = int(m.group(1))
            break

    patient = None
    for x in reversed(parts):
        m = _PATIENT_RE.search(x)
        if m:
            patient = m.group(1).upper()
            break

    if patient is None:
        return None
    if binary is None and code is not None:
        binary = 0 if code in ("A", "F", "PT", "TA") else 1
    if binary is None:
        return None

    return {
        "patient_id": patient,
        "binary_label": binary,
        "subtype_code": code or ("F" if binary == 0 else "DC"),
        "subtype_label": SUBTYPE_TO_IDX[code] if code else (1 if binary == 0 else 4),
        "magnification": mag or 0,
        "_inferred": True,
    }


@dataclass(frozen=True)
class ImageRecord:
    path: str
    patient_id: str
    binary_label: int      # 0 benign, 1 malignant
    subtype_code: str
    subtype_label: int
    magnification: int


def parse_filename(name: str) -> dict | None:
    """Parse one BreakHis filename. Returns None if it does not match."""
    m = _FNAME_RE.match(name)
    if m is None:
        return None
    subtype = m.group("subtype").upper()
    if subtype not in SUBTYPE_TO_IDX:
        return None
    cls = m.group("cls").upper()
    return {
        "patient_id": m.group("patient").upper(),
        "binary_label": 0 if cls == "B" else 1,
        "subtype_code": subtype,
        "subtype_label": SUBTYPE_TO_IDX[subtype],
        "magnification": int(m.group("mag")),
    }


#: Directories worth probing when no explicit root is given.
DEFAULT_SEARCH_PATHS = (
    "/kaggle/input",      # Kaggle mounts each dataset under its own slug here
    "/content",           # Google Colab
    "./data", "../data", "./BreaKHis_v1", "../BreaKHis_v1", ".",
)


def inventory(path, max_files: int = 8) -> dict:
    """Describe what is actually in a directory: counts, extensions, samples.

    This turns "not found" into a diagnosis. Reporting only what is missing
    forces a round trip; reporting what is present usually answers the question.
    """
    from collections import Counter

    path = Path(path)
    if not path.exists():
        return {"exists": False}
    try:
        files = [f for f in path.rglob("*") if f.is_file()]
        subdirs = sorted(d.name for d in path.iterdir() if d.is_dir())[:20]
    except (PermissionError, OSError) as e:
        return {"exists": True, "error": str(e)}
    return {
        "exists": True,
        "n_files": len(files),
        "extensions": Counter(f.suffix.lower() for f in files).most_common(8),
        "n_images": sum(1 for f in files if f.suffix.lower() in IMAGE_EXTS),
        "samples": [str(f.relative_to(path)) for f in files[:max_files]],
        "subdirs": subdirs,
    }


def describe_search_locations(candidates=None) -> str:
    """Readable report of every probe location. Safe to call at any time."""
    out = []
    for c in list(candidates or DEFAULT_SEARCH_PATHS):
        inv = inventory(c)
        if not inv["exists"]:
            out.append(f"  {c}\n      does not exist")
        elif "error" in inv:
            out.append(f"  {c}\n      unreadable: {inv['error']}")
        else:
            out.append(
                f"  {c}\n"
                f"      subdirs   : {inv['subdirs'] or '(none)'}\n"
                f"      files     : {inv['n_files']}  (images: {inv['n_images']})\n"
                f"      extensions: {inv['extensions'] or '(none)'}\n"
                f"      samples   : {inv['samples'][:4] or '(none)'}"
            )
    return "\n".join(out)


def find_dataset_root(candidates=None, verbose: bool = True) -> Path:
    """Locate a BreakHis tree without hardcoding a mount path.

    Kaggle mounts every attached dataset under ``/kaggle/input/<slug>``, and
    BreakHis mirrors circulate under different slugs and sometimes with
    restructured folders, so a hardcoded path breaks for anyone who attached a
    different copy.

    Three passes, most specific first:
      1. a file matching the canonical BreakHis filename grammar
      2. any file whose name begins with ``SOB``
      3. any directory holding a substantial number of images

    A pass-3 match is returned with a warning, because :func:`build_index` can
    often still label such a tree from its folder structure.

    On failure the error carries a full inventory of everything searched, so the
    message diagnoses the problem rather than merely reporting it.
    """
    candidates = list(candidates or DEFAULT_SEARCH_PATHS)

    def probes_for(base: Path):
        try:
            kids = [d for d in sorted(base.iterdir()) if d.is_dir()]
        except (PermissionError, OSError):
            kids = []
        return kids + [base]          # children first: prefer the specific folder

    for pattern, label in ((None, "canonical BreakHis filename"),
                           ("SOB*", "SOB-prefixed filename")):
        for c in candidates:
            base = Path(c)
            if not base.exists():
                continue
            for probe in probes_for(base):
                try:
                    if pattern is None:
                        hit = next((f for f in probe.rglob("*")
                                    if f.is_file() and parse_filename(f.name)), None)
                    else:
                        hit = next((f for f in probe.rglob(pattern) if f.is_file()), None)
                except (PermissionError, OSError):
                    continue
                if hit is not None:
                    if verbose:
                        print(f"Found BreakHis under: {probe}")
                        print(f"  matched by : {label}")
                        print(f"  example    : {hit}")
                    return probe

    for c in candidates:
        base = Path(c)
        if not base.exists():
            continue
        for probe in probes_for(base):
            inv = inventory(probe)
            if inv.get("n_images", 0) >= 50:
                import warnings
                warnings.warn(
                    f"No BreakHis-style filenames found, but {probe} holds "
                    f"{inv['n_images']} images. Using it and inferring labels "
                    f"from the folder structure. VERIFY the counts in Table I "
                    f"before trusting them. Samples: {inv['samples'][:3]}",
                    RuntimeWarning,
                )
                if verbose:
                    print(f"Falling back to image-bearing directory: {probe}")
                return probe

    raise FileNotFoundError(
        "Could not locate a BreakHis image tree.\n\n"
        "WHAT IS ACTUALLY PRESENT:\n" + describe_search_locations(candidates) + "\n\n"
        "HOW TO READ THIS:\n"
        "  * /kaggle/input has no subdirs  -> no dataset is attached. Use the\n"
        "    right-hand panel: Add Data -> search 'breakhis' -> add it.\n"
        "  * a subdir is listed but its samples do not start with 'SOB'\n"
        "    -> restructured mirror. Call build_index() on that path directly;\n"
        "    it will try to infer labels from the folder names.\n"
        "  * otherwise pass the directory explicitly: build_index('/path/to/data')\n"
    )


def acquire_via_kagglehub(slug: str = "ambarish/breakhis", verbose: bool = True):
    """Download BreakHis through kagglehub when no copy is already mounted.

    Inside a Kaggle notebook the runtime is already authenticated, so this needs
    no token and no interaction with the Add Data panel -- which is the step that
    most often goes wrong. Requires Internet to be enabled in notebook settings.

    Returns the download path, or None if the attempt fails for any reason. The
    caller decides what to do about it; this never raises.
    """
    try:
        import kagglehub
    except ImportError:
        try:
            import subprocess, sys as _sys
            subprocess.run([_sys.executable, "-m", "pip", "install", "-q", "kagglehub"],
                           check=True)
            import kagglehub
        except Exception as e:
            if verbose:
                print(f"kagglehub unavailable ({e}); cannot auto-download.")
            return None

    try:
        if verbose:
            print(f"No local copy found. Downloading '{slug}' via kagglehub "
                  f"(a few minutes, ~4 GB)...")
        path = kagglehub.dataset_download(slug)
        if verbose:
            print(f"Downloaded to: {path}")
        return Path(path)
    except Exception as e:
        if verbose:
            print(f"kagglehub download failed: {type(e).__name__}: {e}")
            print("Most likely cause: Internet is OFF in notebook settings "
                  "(Settings -> Internet -> On).")
        return None


def locate_or_acquire(candidates=None, slug: str = "ambarish/breakhis",
                      verbose: bool = True) -> Path:
    """Find a mounted BreakHis copy, or download one if none is present.

    This is the entry point the notebook uses, so a missing dataset resolves
    itself instead of stopping the run.
    """
    try:
        return find_dataset_root(candidates, verbose=verbose)
    except FileNotFoundError as first_error:
        if verbose:
            print("No mounted copy found; attempting download.\n")
        got = acquire_via_kagglehub(slug, verbose=verbose)
        if got is None:
            raise first_error
        try:
            return find_dataset_root([str(got)], verbose=verbose)
        except FileNotFoundError:
            return Path(got)


def build_index(root: str | Path | None = None) -> pd.DataFrame:
    """Walk the BreakHis tree and build a dataframe of every parsable image.

    Pass root=None to auto-discover the dataset (see find_dataset_root).

    Works regardless of intermediate directory layout — it keys entirely off the
    filename grammar, so it tolerates the several folder arrangements the
    dataset circulates in (official tarball, Kaggle mirrors, flattened copies).
    """
    if root is None:
        root = find_dataset_root()
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(
            f"BreakHis root does not exist: {root}\n"
            "Call build_index() with no argument to auto-discover it, or see "
            "find_dataset_root() for the locations it probes."
        )

    images = [f for f in root.rglob("*")
              if f.is_file() and f.suffix.lower() in IMAGE_EXTS]

    rows, skipped = [], 0
    for f in images:
        meta = parse_filename(f.name)
        if meta is None:
            skipped += 1
            continue
        meta["_inferred"] = False
        meta["path"] = str(f)
        rows.append(meta)

    n_by_filename = len(rows)

    # Fallback: a restructured mirror whose folders still encode the labels.
    if n_by_filename == 0 and images:
        import warnings
        inferred, unrecoverable = [], 0
        for f in images:
            meta = infer_from_path(f, root)
            if meta is None:
                unrecoverable += 1
                continue
            meta["path"] = str(f)
            inferred.append(meta)

        if not inferred:
            raise RuntimeError(
                f"Found {len(images)} images under {root} but could not recover "
                f"labels from either filenames or folder names.\n"
                f"Sample paths:\n  " +
                "\n  ".join(str(f.relative_to(root)) for f in images[:5]) +
                "\n\nThis mirror is not usable for a patient-wise study: without "
                "slide identifiers there is no way to keep a patient out of both "
                "training and test. Use a mirror that preserves the original "
                "BreaKHis_v1 filenames, e.g. the Kaggle dataset 'ambarish/breakhis'."
            )

        frac = len(inferred) / len(images)
        warnings.warn(
            f"Filenames did not match the BreaKHis grammar. Labels were inferred "
            f"from folder structure for {len(inferred)}/{len(images)} images "
            f"({frac:.1%}); {unrecoverable} had no recoverable patient id and were "
            f"dropped. VERIFY the counts in the dataset table before proceeding, "
            f"and state in the paper that a restructured mirror was used.",
            RuntimeWarning,
        )
        rows, skipped = inferred, unrecoverable

    if not rows:
        raise RuntimeError(
            f"No BreakHis images parsed under {root}.\n"
            f"Images found: {len(images)}\n"
            "Check that the path points at an extracted BreaKHis tree. "
            "Call data.describe_search_locations() to see what is present."
        )

    df = pd.DataFrame(rows)
    # Deduplicate: some mirrors ship the same image under two folder paths.
    df["_key"] = (
        df.subtype_code + "|" + df.patient_id + "|"
        + df.magnification.astype(str) + "|"
        + df.path.map(lambda s: Path(s).name)
    )
    before = len(df)
    df = df.drop_duplicates("_key").drop(columns="_key").reset_index(drop=True)
    df.attrs["n_duplicates_removed"] = before - len(df)
    df.attrs["n_unparsable_files"] = skipped
    df.attrs["labels_inferred_from_path"] = bool(df["_inferred"].any()) if "_inferred" in df else False
    if "_inferred" in df:
        df = df.drop(columns="_inferred")
    return df


def verify_against_published(df: pd.DataFrame) -> pd.DataFrame:
    """Compare the parsed index against the counts published by Spanhol et al.

    Returns a comparison table. Deviations are surfaced rather than raised: a
    Kaggle mirror may legitimately be a subset, but the paper must then say so.
    """
    published = {
        ("benign", "A"): 444, ("benign", "F"): 1014,
        ("benign", "PT"): 453, ("benign", "TA"): 569,
        ("malignant", "DC"): 3451, ("malignant", "LC"): 626,
        ("malignant", "MC"): 792, ("malignant", "PC"): 560,
    }
    out = []
    for (cls, code), expected in published.items():
        want_bin = 0 if cls == "benign" else 1
        got = int(((df.binary_label == want_bin) & (df.subtype_code == code)).sum())
        out.append({
            "class": cls,
            "subtype": SUBTYPE_NAMES[code],
            "published": expected,
            "found": got,
            "delta": got - expected,
            "match": got == expected,
        })
    tbl = pd.DataFrame(out)
    tbl.loc[len(tbl)] = {
        "class": "TOTAL", "subtype": "",
        "published": 7909, "found": int(len(df)),
        "delta": int(len(df)) - 7909, "match": int(len(df)) == 7909,
    }
    return tbl


def make_patient_wise_folds(
    df: pd.DataFrame,
    n_splits: int = 5,
    label_col: str = "binary_label",
    seed: int = 42,
) -> pd.DataFrame:
    """Assign a fold index to every image, grouping by patient.

    StratifiedGroupKFold keeps whole patients together while balancing the class
    ratio across folds as far as the grouping permits. Adds a 'fold' column.
    """
    df = df.copy()
    splitter = StratifiedGroupKFold(
        n_splits=n_splits, shuffle=True, random_state=seed
    )
    df["fold"] = -1
    for fold, (_, val_idx) in enumerate(
        splitter.split(df, y=df[label_col], groups=df["patient_id"])
    ):
        df.loc[df.index[val_idx], "fold"] = fold

    if (df["fold"] < 0).any():
        raise RuntimeError("Fold assignment left unassigned rows.")
    assert_no_patient_leakage(df)
    return df


def assert_no_patient_leakage(df: pd.DataFrame, fold_col: str = "fold") -> None:
    """Hard guarantee that each patient lives in exactly one fold."""
    per_patient = df.groupby("patient_id")[fold_col].nunique()
    offenders = per_patient[per_patient > 1]
    if len(offenders) > 0:
        raise AssertionError(
            "PATIENT LEAKAGE DETECTED — these patients span multiple folds:\n"
            f"{offenders.to_string()}\n"
            "This invalidates the evaluation. Fix the splitter before training."
        )


def assert_disjoint(train_df: pd.DataFrame, eval_df: pd.DataFrame) -> None:
    """Guarantee a train/eval pair shares no patient. Call before every fit."""
    overlap = set(train_df.patient_id) & set(eval_df.patient_id)
    if overlap:
        raise AssertionError(
            f"PATIENT LEAKAGE: {len(overlap)} patient(s) in both train and eval: "
            f"{sorted(overlap)[:10]}"
        )


def make_image_wise_folds(
    df: pd.DataFrame,
    n_splits: int = 5,
    label_col: str = "binary_label",
    seed: int = 42,
) -> pd.DataFrame:
    """Deliberately leaky image-wise split — the protocol most prior work uses.

    Included ONLY so the paper can quantify the leakage effect by running the
    identical model under both protocols. Never use this to report headline
    performance.
    """
    from sklearn.model_selection import StratifiedKFold

    df = df.copy()
    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    df["fold"] = -1
    for fold, (_, val_idx) in enumerate(splitter.split(df, y=df[label_col])):
        df.loc[df.index[val_idx], "fold"] = fold
    return df


def split_summary(df: pd.DataFrame) -> pd.DataFrame:
    """Per-fold composition table for the paper's protocol section."""
    g = df.groupby("fold")
    return pd.DataFrame({
        "n_images": g.size(),
        "n_patients": g["patient_id"].nunique(),
        "n_benign": g["binary_label"].apply(lambda s: int((s == 0).sum())),
        "n_malignant": g["binary_label"].apply(lambda s: int((s == 1).sum())),
        "pct_malignant": g["binary_label"].mean().mul(100).round(2),
    }).reset_index()


Writing /kaggle/working/src/data.py


In [ ]:
%%writefile /kaggle/working/src/models.py
"""
Architectures: a cross-attention CNN-Transformer hybrid plus matched baselines.

Design rationale for the hybrid (BCAF — Bidirectional Cross-Attention Fusion):
histopathology decisions depend jointly on local cellular morphology (nuclear
pleomorphism, mitotic figures) and on global tissue architecture (glandular
organisation, stromal pattern). A CNN encodes the former with a strong locality
prior; a windowed transformer encodes the latter. Prior hybrids concatenate the
two streams or apply one-directional attention. BCAF lets each stream query the
other, so convolutional features are re-weighted by architectural context and
transformer tokens are re-weighted by cytological evidence, before a gated sum
decides how much each stream contributes per-sample.

Every model exposes the same interface so baselines and hybrid are trained,
calibrated and explained by identical code paths — a precondition for the
statistical comparisons this paper makes.
"""
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import timm
    _HAS_TIMM = True
except ImportError:  # pragma: no cover
    _HAS_TIMM = False


# ----------------------------------------------------------------------------
# Backbone construction
# ----------------------------------------------------------------------------
def _is_transformer(name: str) -> bool:
    return any(k in name for k in ("swin", "vit", "deit", "beit", "cait"))


def _make_backbone(name: str, pretrained: bool = True, img_size: int = 224):
    """Return (module, feature_dim, kind) where kind is 'cnn' or 'transformer'.

    Backbones are created with num_classes=0 and global_pool='' so they emit
    spatial feature maps / token sequences rather than pooled logits.

    Windowed transformers (Swin) and patch transformers (ViT) have a fixed
    input resolution baked into their positional embeddings and window
    partitioning. We forward img_size to timm so a non-default resolution is
    handled at construction time, and fail with an actionable message rather
    than an opaque assertion deep inside the forward pass.
    """
    if not _HAS_TIMM:
        raise ImportError("timm is required. pip install timm")

    kwargs = dict(pretrained=pretrained, num_classes=0, global_pool="")
    if _is_transformer(name) and img_size != 224:
        kwargs["img_size"] = img_size
    try:
        m = timm.create_model(name, **kwargs)
    except (TypeError, RuntimeError, AssertionError) as e:
        raise ValueError(
            f"Backbone '{name}' cannot be built at img_size={img_size}. "
            f"Windowed transformers need a resolution divisible by "
            f"patch_size x window_size (e.g. 224 for swin_tiny_patch4_window7). "
            f"Use img_size=224, or choose a CNN backbone. Original error: {e}"
        ) from e

    dim = m.num_features
    return m, dim, ("transformer" if _is_transformer(name) else "cnn")


def _to_tokens(feat: torch.Tensor) -> torch.Tensor:
    """Normalise any backbone output to (B, N, C) token form.

    Handles the three shapes timm backbones emit:
      (B, C, H, W) -> CNN / ConvNeXt feature map
      (B, H, W, C) -> Swin in channels-last layout
      (B, N, C)    -> ViT token sequence
    """
    if feat.dim() == 4:
        b, d1, d2, d3 = feat.shape
        # channels-last if the last dim is the largest and spatial dims match
        if d1 == d2 and d3 != d1:
            return feat.reshape(b, d1 * d2, d3)          # (B,H,W,C)
        return feat.flatten(2).transpose(1, 2)            # (B,C,H,W)
    if feat.dim() == 3:
        return feat
    raise ValueError(f"Unsupported feature shape {tuple(feat.shape)}")


# ----------------------------------------------------------------------------
# Cross-attention fusion
# ----------------------------------------------------------------------------
class CrossAttentionBlock(nn.Module):
    """One direction of cross-attention: queries from x, keys/values from y."""

    def __init__(self, dim: int, num_heads: int = 8, mlp_ratio: float = 2.0,
                 dropout: float = 0.1):
        super().__init__()
        self.norm_q = nn.LayerNorm(dim)
        self.norm_kv = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(
            dim, num_heads, dropout=dropout, batch_first=True
        )
        self.norm_mlp = nn.LayerNorm(dim)
        hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden, dim),
        )
        self.drop = nn.Dropout(dropout)
        self._last_attn = None

    def forward(self, x: torch.Tensor, y: torch.Tensor,
                store_attn: bool = False) -> torch.Tensor:
        q = self.norm_q(x)
        kv = self.norm_kv(y)
        out, w = self.attn(q, kv, kv, need_weights=store_attn,
                           average_attn_weights=True)
        if store_attn:
            self._last_attn = w.detach()
        x = x + self.drop(out)
        x = x + self.drop(self.mlp(self.norm_mlp(x)))
        return x

    @property
    def last_attention(self):
        """(B, Nq, Nkv) attention map from the most recent forward pass.

        Used by the XAI module to produce fusion-level explanations that are
        native to the architecture rather than retro-fitted.
        """
        return self._last_attn


class BCAFNet(nn.Module):
    """Bidirectional Cross-Attention Fusion network.

    Args:
        cnn_name: timm CNN backbone (local/cytological stream).
        vit_name: timm transformer backbone (global/architectural stream).
        num_classes: 2 for benign/malignant, 8 for subtype.
        fusion_dim: shared projection width for the two streams.
    """

    def __init__(
        self,
        cnn_name: str = "convnext_tiny",
        vit_name: str = "swin_tiny_patch4_window7_224",
        num_classes: int = 2,
        fusion_dim: int = 384,
        num_heads: int = 8,
        dropout: float = 0.1,
        pretrained: bool = True,
        img_size: int = 224,
    ):
        super().__init__()
        self.img_size = img_size
        self.cnn, cnn_dim, _ = _make_backbone(cnn_name, pretrained, img_size)
        self.vit, vit_dim, _ = _make_backbone(vit_name, pretrained, img_size)

        self.proj_cnn = nn.Linear(cnn_dim, fusion_dim)
        self.proj_vit = nn.Linear(vit_dim, fusion_dim)

        self.c2v = CrossAttentionBlock(fusion_dim, num_heads, dropout=dropout)
        self.v2c = CrossAttentionBlock(fusion_dim, num_heads, dropout=dropout)

        # Per-sample gate: learns how much each stream should contribute.
        # Reported in the paper as evidence of what the fusion actually does.
        self.gate = nn.Sequential(
            nn.Linear(fusion_dim * 2, fusion_dim // 2), nn.GELU(),
            nn.Linear(fusion_dim // 2, 2),
        )
        self.norm = nn.LayerNorm(fusion_dim)
        self.head = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(fusion_dim, num_classes)
        )
        self._last_gate = None

    def forward(self, x: torch.Tensor, store_attn: bool = False) -> torch.Tensor:
        if x.shape[-1] != self.img_size or x.shape[-2] != self.img_size:
            raise ValueError(
                f"BCAFNet was built for {self.img_size}x{self.img_size} inputs "
                f"but received {tuple(x.shape[-2:])}. Rebuild the model with "
                f"img_size={x.shape[-1]}, or set TrainConfig.img_size="
                f"{self.img_size}."
            )
        tc = self.proj_cnn(_to_tokens(self.cnn(x)))   # (B, Nc, D)
        tv = self.proj_vit(_to_tokens(self.vit(x)))   # (B, Nv, D)

        fc = self.v2c(tc, tv, store_attn)   # conv tokens attend to global context
        fv = self.c2v(tv, tc, store_attn)   # global tokens attend to local evidence

        pc, pv = fc.mean(1), fv.mean(1)
        w = F.softmax(self.gate(torch.cat([pc, pv], dim=-1)), dim=-1)
        self._last_gate = w.detach()
        fused = w[:, 0:1] * pc + w[:, 1:2] * pv
        return self.head(self.norm(fused))

    @property
    def last_gate(self):
        """(B, 2) softmax weights over [cnn_stream, transformer_stream]."""
        return self._last_gate


class SingleBackboneNet(nn.Module):
    """Baseline wrapper giving any timm backbone the same interface as BCAFNet."""

    def __init__(self, name: str, num_classes: int = 2,
                 dropout: float = 0.1, pretrained: bool = True,
                 img_size: int | None = None):
        super().__init__()
        if not _HAS_TIMM:
            raise ImportError("timm is required. pip install timm")
        kw = dict(pretrained=pretrained, num_classes=0)
        if img_size is not None and _is_transformer(name) and img_size != 224:
            kw["img_size"] = img_size
        self.backbone = timm.create_model(name, **kw)
        self.head = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(self.backbone.num_features, num_classes)
        )
        self.name = name

    def forward(self, x: torch.Tensor, store_attn: bool = False) -> torch.Tensor:
        return self.head(self.backbone(x))


class ConcatFusionNet(nn.Module):
    """Ablation: same two streams, naive concatenation instead of cross-attention.

    This is the control that isolates whether cross-attention contributes
    anything beyond simply having two backbones — the ablation most prior
    hybrid papers omit, and the one reviewers ask for.
    """

    def __init__(self, cnn_name: str = "convnext_tiny",
                 vit_name: str = "swin_tiny_patch4_window7_224",
                 num_classes: int = 2, dropout: float = 0.1,
                 pretrained: bool = True, img_size: int = 224):
        super().__init__()
        self.img_size = img_size
        self.cnn, cd, _ = _make_backbone(cnn_name, pretrained, img_size)
        self.vit, vd, _ = _make_backbone(vit_name, pretrained, img_size)
        self.head = nn.Sequential(
            nn.LayerNorm(cd + vd), nn.Dropout(dropout),
            nn.Linear(cd + vd, num_classes),
        )

    def forward(self, x: torch.Tensor, store_attn: bool = False) -> torch.Tensor:
        c = _to_tokens(self.cnn(x)).mean(1)
        v = _to_tokens(self.vit(x)).mean(1)
        return self.head(torch.cat([c, v], dim=-1))


# ----------------------------------------------------------------------------
# Registry
# ----------------------------------------------------------------------------
MODEL_REGISTRY = {
    "bcaf":            lambda nc, pt, sz: BCAFNet(num_classes=nc, pretrained=pt, img_size=sz),
    "concat_ablation": lambda nc, pt, sz: ConcatFusionNet(num_classes=nc, pretrained=pt, img_size=sz),
    "resnet50":        lambda nc, pt, sz: SingleBackboneNet("resnet50", nc, pretrained=pt),
    "densenet121":     lambda nc, pt, sz: SingleBackboneNet("densenet121", nc, pretrained=pt),
    "efficientnet_b0": lambda nc, pt, sz: SingleBackboneNet("efficientnet_b0", nc, pretrained=pt),
    "convnext_tiny":   lambda nc, pt, sz: SingleBackboneNet("convnext_tiny", nc, pretrained=pt),
    "swin_tiny":       lambda nc, pt, sz: SingleBackboneNet("swin_tiny_patch4_window7_224", nc, pretrained=pt, img_size=sz),
    "vit_small":       lambda nc, pt, sz: SingleBackboneNet("vit_small_patch16_224", nc, pretrained=pt, img_size=sz),
}

#: Models whose input resolution is fixed by construction.
FIXED_RESOLUTION_MODELS = ("bcaf", "concat_ablation", "swin_tiny", "vit_small")


def build_model(key: str, num_classes: int = 2, pretrained: bool = True,
                img_size: int = 224) -> nn.Module:
    if key not in MODEL_REGISTRY:
        raise KeyError(f"Unknown model '{key}'. Available: {sorted(MODEL_REGISTRY)}")
    return MODEL_REGISTRY[key](num_classes, pretrained, img_size)


def count_parameters(model: nn.Module) -> dict:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total_M": round(total / 1e6, 2), "trainable_M": round(trainable / 1e6, 2)}

Writing /kaggle/working/src/models.py


In [ ]:
%%writefile /kaggle/working/src/engine.py
"""
Training and evaluation engine.

All models pass through identical code here — same optimiser family, schedule,
augmentation, early stopping and evaluation — so that any measured difference
between architectures is attributable to the architecture rather than to
training-recipe differences. This is a precondition for the paper's claims.
"""
from __future__ import annotations

import copy
import time
from dataclasses import dataclass, field, asdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             cohen_kappa_score, confusion_matrix, f1_score,
                             matthews_corrcoef, precision_score, recall_score,
                             roc_auc_score)
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


@dataclass
class TrainConfig:
    model_key: str = "bcaf"
    num_classes: int = 2
    img_size: int = 224
    batch_size: int = 32
    epochs: int = 30
    lr: float = 3e-4
    backbone_lr_mult: float = 0.1     # discriminative fine-tuning
    weight_decay: float = 0.05
    label_smoothing: float = 0.05
    warmup_epochs: int = 3
    patience: int = 8
    amp: bool = True
    num_workers: int = 2
    seed: int = 42
    class_balanced_loss: bool = True
    task: str = "binary"              # or "subtype"
    protocol: str = "patient_wise"    # or "image_wise"
    magnification: str = "all"        # "all" | "40" | "100" | "200" | "400"
    extra: dict = field(default_factory=dict)


def set_seed(seed: int) -> None:
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ----------------------------------------------------------------------------
# Data
# ----------------------------------------------------------------------------
def build_transforms(img_size: int, train: bool):
    """Stain-aware augmentation.

    Colour jitter on hue/saturation is the important one for H&E: it simulates
    the staining and scanner variation that is the dominant domain shift between
    pathology labs, and is what a model must be invariant to if it is to
    generalise beyond the acquisition site.
    """
    if train:
        return transforms.Compose([
            transforms.RandomResizedCrop(img_size, scale=(0.7, 1.0), ratio=(0.85, 1.18)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomApply([transforms.RandomRotation(20)], p=0.5),
            transforms.ColorJitter(brightness=0.2, contrast=0.2,
                                   saturation=0.25, hue=0.05),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
            transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),
        ])
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


class BreakHisDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform, label_col: str = "binary_label"):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.label_col = label_col

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, i: int):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")
        return self.transform(img), int(row[self.label_col]), i


def make_loaders(train_df, val_df, cfg: TrainConfig, label_col: str):
    tr = BreakHisDataset(train_df, build_transforms(cfg.img_size, True), label_col)
    va = BreakHisDataset(val_df, build_transforms(cfg.img_size, False), label_col)
    return (
        DataLoader(tr, batch_size=cfg.batch_size, shuffle=True,
                   num_workers=cfg.num_workers, pin_memory=True, drop_last=True),
        DataLoader(va, batch_size=cfg.batch_size * 2, shuffle=False,
                   num_workers=cfg.num_workers, pin_memory=True),
    )


# ----------------------------------------------------------------------------
# Metrics
# ----------------------------------------------------------------------------
def compute_metrics(y_true, y_pred, probs, num_classes: int) -> dict:
    m = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "kappa": cohen_kappa_score(y_true, y_pred),
    }
    try:
        if num_classes == 2:
            m["auc"] = roc_auc_score(y_true, probs[:, 1])
            tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
            m["sensitivity"] = tp / (tp + fn) if (tp + fn) else 0.0
            m["specificity"] = tn / (tn + fp) if (tn + fp) else 0.0
            # Clinically the decisive number: a missed malignancy is the costly error.
            m["false_negative_rate"] = fn / (fn + tp) if (fn + tp) else 0.0
        else:
            m["auc"] = roc_auc_score(y_true, probs, multi_class="ovr",
                                     average="macro")
    except (ValueError, IndexError):
        m["auc"] = float("nan")
    return m


def aggregate_to_patient(df_idx: pd.DataFrame, probs: np.ndarray,
                        y_true: np.ndarray) -> dict:
    """Patient-level decision by mean probability over that patient's images.

    This is the clinically meaningful unit — a diagnosis is rendered per patient,
    not per field of view — and is almost never reported on BreakHis.
    """
    tmp = pd.DataFrame({"patient_id": df_idx["patient_id"].values, "y": y_true})
    for c in range(probs.shape[1]):
        tmp[f"p{c}"] = probs[:, c]
    g = tmp.groupby("patient_id")
    pcols = [f"p{c}" for c in range(probs.shape[1])]
    pat_probs = g[pcols].mean().values
    pat_true = g["y"].first().values
    pat_pred = pat_probs.argmax(axis=1)
    out = compute_metrics(pat_true, pat_pred, pat_probs, probs.shape[1])
    out["n_patients"] = int(len(pat_true))
    return out


# ----------------------------------------------------------------------------
# Train / evaluate
# ----------------------------------------------------------------------------
def _param_groups(model, cfg: TrainConfig):
    head_kw = ("head", "gate", "proj_", "c2v", "v2c", "norm")
    head, backbone = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (head if any(k in n for k in head_kw) else backbone).append(p)
    return [
        {"params": backbone, "lr": cfg.lr * cfg.backbone_lr_mult},
        {"params": head, "lr": cfg.lr},
    ]


@torch.no_grad()
def evaluate(model, loader, device, num_classes: int):
    """Returns (logits, probs, y_true, order_index)."""
    model.eval()
    L, Y, I = [], [], []
    for x, y, idx in loader:
        x = x.to(device, non_blocking=True)
        L.append(model(x).float().cpu())
        Y.append(y)
        I.append(idx)
    logits = torch.cat(L)
    y = torch.cat(Y).numpy()
    idx = torch.cat(I).numpy()
    probs = F.softmax(logits, dim=1).numpy()
    return logits.numpy(), probs, y, idx




def train_fold(model, train_df, val_df, cfg: TrainConfig, device,
              label_col: str = "binary_label", verbose: bool = True):
    set_seed(cfg.seed)
    train_loader, val_loader = make_loaders(train_df, val_df, cfg, label_col)

    if cfg.class_balanced_loss:
        counts = np.bincount(train_df[label_col].values, minlength=cfg.num_classes)
        w = torch.tensor(counts.sum() / (cfg.num_classes * np.maximum(counts, 1)),
                         dtype=torch.float32, device=device)
    else:
        w = None
    criterion = nn.CrossEntropyLoss(weight=w, label_smoothing=cfg.label_smoothing)

    model = model.to(device)
    opt = torch.optim.AdamW(_param_groups(model, cfg), weight_decay=cfg.weight_decay)
    steps = max(1, len(train_loader))
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=[g["lr"] for g in opt.param_groups],
        total_steps=cfg.epochs * steps,
        pct_start=min(0.4, cfg.warmup_epochs / max(cfg.epochs, 1)),
    )
    use_amp = cfg.amp and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    best = {"score": -np.inf, "epoch": -1, "state": None}
    history, since_improve = [], 0

    for epoch in range(cfg.epochs):
        model.train()
        t0, run_loss, seen = time.time(), 0.0, 0
        for x, y, _ in train_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=use_amp):
                loss = criterion(model(x), y)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            run_loss += loss.item() * x.size(0)
            seen += x.size(0)

        _, probs, y_true, _ = evaluate(model, val_loader, device, cfg.num_classes)
        met = compute_metrics(y_true, probs.argmax(1), probs, cfg.num_classes)
        met.update(epoch=epoch, train_loss=run_loss / max(seen, 1),
                   seconds=round(time.time() - t0, 1))
        history.append(met)

        # Balanced accuracy as the selection criterion — BreakHis is ~2:1
        # malignant, so plain accuracy rewards majority-class bias.
        score = met["balanced_accuracy"]
        if score > best["score"]:
            best = {"score": score, "epoch": epoch,
                    "state": copy.deepcopy(model.state_dict())}
            since_improve = 0
        else:
            since_improve += 1

        if verbose:
            print(f"  ep {epoch:02d} loss {met['train_loss']:.4f} "
                  f"acc {met['accuracy']:.4f} bal {score:.4f} "
                  f"auc {met['auc']:.4f} ({met['seconds']}s)"
                  + ("  <- best" if since_improve == 0 else ""))
        if since_improve >= cfg.patience:
            if verbose:
                print(f"  early stop at epoch {epoch} "
                      f"(best epoch {best['epoch']}, bal {best['score']:.4f})")
            break

    if best["state"] is not None:
        model.load_state_dict(best["state"])
    return model, pd.DataFrame(history), best


def run_cross_validation(df, cfg: TrainConfig, build_fn, device,
                        label_col: str = "binary_label", verbose: bool = True,
                        ckpt_dir: str | None = None):
    """Full k-fold run. Returns per-fold metrics and stacked out-of-fold outputs.

    The out-of-fold predictions are what every downstream analysis uses —
    statistical tests, calibration, XAI faithfulness — so each sample is scored
    exactly once, by a model that never saw that patient.

    If ckpt_dir is given, each fold's best weights are written there. The
    explanation analysis then loads them instead of retraining, which on a
    six-model grid saves several GPU-hours and — more importantly — guarantees
    the explanations are generated from the exact weights that produced the
    reported metrics, rather than from a second run that merely used the same
    recipe.
    """
    import os
    from data import assert_disjoint

    if ckpt_dir:
        os.makedirs(ckpt_dir, exist_ok=True)

    folds = sorted(df["fold"].unique())
    rows, oof, ckpts = [], [], []
    for f in folds:
        tr = df[df.fold != f]
        va = df[df.fold == f]
        if cfg.protocol == "patient_wise":
            assert_disjoint(tr, va)
        if verbose:
            print(f"\n--- fold {f}: train {len(tr)} imgs / "
                  f"{tr.patient_id.nunique()} pts | "
                  f"val {len(va)} imgs / {va.patient_id.nunique()} pts ---")

        model = build_fn()
        model, hist, best = train_fold(model, tr, va, cfg, device, label_col, verbose)

        if ckpt_dir:
            cp = os.path.join(ckpt_dir, f"{cfg.model_key}_fold{f}.pt")
            torch.save({"state_dict": model.state_dict(),
                        "model_key": cfg.model_key,
                        "num_classes": cfg.num_classes,
                        "img_size": cfg.img_size,
                        "fold": int(f),
                        "best_epoch": best["epoch"],
                        "best_score": best["score"]}, cp)
            ckpts.append(cp)
            if verbose:
                print(f"  checkpoint -> {cp}")

        _, val_loader = make_loaders(tr, va, cfg, label_col)
        logits, probs, y_true, order = evaluate(model, val_loader, device,
                                                cfg.num_classes)
        va_ord = va.reset_index(drop=True).iloc[order]

        img_m = compute_metrics(y_true, probs.argmax(1), probs, cfg.num_classes)
        pat_m = aggregate_to_patient(va_ord, probs, y_true)
        rows.append({"fold": f, "best_epoch": best["epoch"],
                     **{f"img_{k}": v for k, v in img_m.items()},
                     **{f"pat_{k}": v for k, v in pat_m.items()}})

        oof.append(pd.DataFrame({
            "fold": f,
            "path": va_ord["path"].values,
            "patient_id": va_ord["patient_id"].values,
            "magnification": va_ord["magnification"].values,
            "y_true": y_true,
            "y_pred": probs.argmax(1),
            "correct": (probs.argmax(1) == y_true).astype(int),
            **{f"prob_{c}": probs[:, c] for c in range(probs.shape[1])},
            **{f"logit_{c}": logits[:, c] for c in range(logits.shape[1])},
        }))
        del model
        if device.type == "cuda":
            torch.cuda.empty_cache()

    result = pd.DataFrame(rows), pd.concat(oof, ignore_index=True)
    if ckpt_dir:
        return (*result, ckpts)
    return result


def load_checkpoint(path: str, build_fn, device):
    """Restore a fold's trained weights saved by run_cross_validation.

    Used by the explanation analysis so that saliency is computed from the very
    weights that produced the reported metrics.
    """
    blob = torch.load(path, map_location=device, weights_only=False)
    model = build_fn()
    model.load_state_dict(blob["state_dict"])
    return model.to(device).eval(), blob


def summarise_cv(fold_df: pd.DataFrame) -> pd.DataFrame:
    """mean ± std across folds, formatted for a results table."""
    num = fold_df.select_dtypes(include=[np.number]).drop(columns=["fold"],
                                                          errors="ignore")
    return pd.DataFrame({"mean": num.mean(), "std": num.std(ddof=1)}).round(4)

Writing /kaggle/working/src/engine.py


In [ ]:
%%writefile /kaggle/working/src/calibration.py
"""
Calibration analysis — how well predicted probabilities match observed frequency.

Accuracy alone cannot tell you whether a model is deployable. A model that is
92% accurate but assigns 0.99 confidence to its errors is more dangerous in a
diagnostic workflow than one that is 90% accurate and correctly signals its own
uncertainty. Calibration is essentially unreported in the BreakHis literature;
this module supplies it.
"""
from __future__ import annotations

import numpy as np

# Torch is needed only for TemperatureScaler. The metrics below are pure NumPy,
# so calibration analysis runs on a machine with no deep-learning stack — which
# is where results are usually analysed and written up.
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    _HAS_TORCH = True
except ImportError:
    torch = nn = F = None
    _HAS_TORCH = False

_Base = nn.Module if _HAS_TORCH else object


def expected_calibration_error(probs: np.ndarray, labels: np.ndarray,
                              n_bins: int = 15) -> float:
    """Equal-width-bin ECE (Guo et al., 2017).

    probs: (N, C) predicted probabilities. labels: (N,) integer ground truth.
    """
    conf = probs.max(axis=1)
    pred = probs.argmax(axis=1)
    correct = (pred == labels).astype(float)

    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    n = len(labels)
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (conf > lo) & (conf <= hi)
        if mask.sum() == 0:
            continue
        ece += (mask.sum() / n) * abs(correct[mask].mean() - conf[mask].mean())
    return float(ece)


def adaptive_calibration_error(probs: np.ndarray, labels: np.ndarray,
                              n_bins: int = 15) -> float:
    """Equal-mass-bin ECE. More stable than equal-width when confidence is
    concentrated near 1.0, which is the usual regime for these models."""
    conf = probs.max(axis=1)
    pred = probs.argmax(axis=1)
    correct = (pred == labels).astype(float)

    order = np.argsort(conf)
    conf, correct = conf[order], correct[order]
    ace, n = 0.0, len(labels)
    for chunk_c, chunk_a in zip(np.array_split(conf, n_bins),
                                np.array_split(correct, n_bins)):
        if len(chunk_c) == 0:
            continue
        ace += (len(chunk_c) / n) * abs(chunk_a.mean() - chunk_c.mean())
    return float(ace)


def maximum_calibration_error(probs: np.ndarray, labels: np.ndarray,
                             n_bins: int = 15) -> float:
    """Worst-case bin gap — the number that matters for safety arguments."""
    conf = probs.max(axis=1)
    correct = (probs.argmax(axis=1) == labels).astype(float)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    gaps = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum() > 0:
            gaps.append(abs(correct[m].mean() - conf[m].mean()))
    return float(max(gaps)) if gaps else 0.0


def brier_score(probs: np.ndarray, labels: np.ndarray) -> float:
    """Multiclass Brier score — a strictly proper scoring rule.

    Unlike ECE it penalises both miscalibration and poor discrimination, so it
    cannot be gamed by a model that is uninformative but well calibrated.
    """
    onehot = np.zeros_like(probs)
    onehot[np.arange(len(labels)), labels] = 1.0
    return float(np.mean(np.sum((probs - onehot) ** 2, axis=1)))


def negative_log_likelihood(probs: np.ndarray, labels: np.ndarray,
                           eps: float = 1e-12) -> float:
    p = np.clip(probs[np.arange(len(labels)), labels], eps, 1.0)
    return float(-np.mean(np.log(p)))


def reliability_curve(probs: np.ndarray, labels: np.ndarray, n_bins: int = 15):
    """Return (bin_centres, observed_accuracy, mean_confidence, bin_counts)."""
    conf = probs.max(axis=1)
    correct = (probs.argmax(axis=1) == labels).astype(float)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    centres, acc, avg_conf, counts = [], [], [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        centres.append((lo + hi) / 2)
        counts.append(int(m.sum()))
        acc.append(float(correct[m].mean()) if m.sum() else np.nan)
        avg_conf.append(float(conf[m].mean()) if m.sum() else np.nan)
    return (np.array(centres), np.array(acc), np.array(avg_conf), np.array(counts))


class TemperatureScaler(_Base):
    """Single-parameter post-hoc calibration (Guo et al., 2017).

    CRITICAL: must be fitted on a validation split that is patient-disjoint from
    both the training data and the test data. Fitting on test leaks.
    """

    def __init__(self):
        if not _HAS_TORCH:
            raise ImportError(
                "TemperatureScaler requires PyTorch (pip install torch). "
                "The calibration metrics in this module work without it."
            )
        super().__init__()
        self.log_temperature = nn.Parameter(torch.zeros(1))

    @property
    def temperature(self) -> float:
        return float(self.log_temperature.exp().item())

    def forward(self, logits: torch.Tensor) -> torch.Tensor:
        return logits / self.log_temperature.exp()

    def fit(self, logits: torch.Tensor, labels: torch.Tensor,
            max_iter: int = 200) -> "TemperatureScaler":
        logits = logits.detach().float()
        labels = labels.detach().long()
        opt = torch.optim.LBFGS([self.log_temperature], lr=0.05, max_iter=max_iter)

        def closure():
            opt.zero_grad()
            loss = F.cross_entropy(self.forward(logits), labels)
            loss.backward()
            return loss

        opt.step(closure)

        t = self.temperature
        if not (0.2 < t < 10.0):
            import warnings
            warnings.warn(
                f"Temperature fitted to {t:.4f}, which is outside the plausible "
                "range (0.2, 10). This usually means the calibration split is "
                "perfectly separated or too small for a stable fit. Report the "
                "raw ECE alongside the scaled one rather than relying on this.",
                RuntimeWarning,
            )
        return self


def calibration_report(probs: np.ndarray, labels: np.ndarray,
                      n_bins: int = 15) -> dict:
    """All calibration metrics in one dict, ready for the results table."""
    return {
        "ECE": expected_calibration_error(probs, labels, n_bins),
        "ACE": adaptive_calibration_error(probs, labels, n_bins),
        "MCE": maximum_calibration_error(probs, labels, n_bins),
        "Brier": brier_score(probs, labels),
        "NLL": negative_log_likelihood(probs, labels),
        "mean_confidence": float(probs.max(axis=1).mean()),
        "accuracy": float((probs.argmax(axis=1) == labels).mean()),
    }

Writing /kaggle/working/src/calibration.py


In [ ]:
%%writefile /kaggle/working/src/xai.py
"""
Explanation generation and — the part the BreakHis literature is missing —
quantitative evaluation of whether those explanations are faithful.

Published BreakHis work shows Grad-CAM heatmaps as qualitative figures. A
heatmap that looks plausible to a reader is not evidence that the model used
that region. This module measures it:

  Deletion AUC   (lower better)  progressively erase the highest-attributed
                                 pixels; a faithful map makes confidence fall fast
  Insertion AUC  (higher better) reveal them onto a blurred canvas; a faithful
                                 map restores confidence fast
  AOPC           (higher better) mean confidence drop over the deletion curve
  Average Drop   (lower better)  confidence lost when only the map is kept
  Coherency      (higher better) agreement between the map and the map-of-the-map
  Complexity     (lower better)  L1 mass of the map — diffuse maps explain little
  ADCC           (higher better) harmonic mean of the three above

References: Petsiuk et al. 2018 (deletion/insertion); Samek et al. 2017 (AOPC);
Poppi et al. 2021 (ADCC).
"""
from __future__ import annotations

import numpy as np
import torch
import torch.nn.functional as F


# ============================================================================
# Attribution methods
# ============================================================================
class GradCAM:
    """Grad-CAM (Selvaraju et al., 2017) with an explicit target layer."""

    def __init__(self, model: torch.nn.Module, target_layer: torch.nn.Module):
        self.model = model
        self.acts = None
        self.grads = None
        self._h1 = target_layer.register_forward_hook(self._save_act)
        self._h2 = target_layer.register_full_backward_hook(self._save_grad)

    def _save_act(self, _m, _i, out):
        self.acts = out[0] if isinstance(out, tuple) else out

    def _save_grad(self, _m, _gi, gout):
        self.grads = gout[0]

    @staticmethod
    def _as_bchw(t: torch.Tensor) -> torch.Tensor:
        """Coerce (B,H,W,C) or (B,N,C) feature maps to (B,C,H,W)."""
        if t.dim() == 4:
            b, d1, d2, d3 = t.shape
            if d1 == d2 and d3 != d1:            # channels-last
                return t.permute(0, 3, 1, 2).contiguous()
            return t
        if t.dim() == 3:                          # token sequence
            b, n, c = t.shape
            s = int(round(n ** 0.5))
            if s * s != n:                        # drop a class token if present
                s = int(round((n - 1) ** 0.5))
                t = t[:, 1:, :]
            return t.transpose(1, 2).reshape(b, c, s, s)
        raise ValueError(f"Cannot interpret feature shape {tuple(t.shape)}")

    def __call__(self, x: torch.Tensor, class_idx=None) -> np.ndarray:
        self.model.zero_grad(set_to_none=True)
        logits = self.model(x)
        if class_idx is None:
            class_idx = logits.argmax(dim=1)
        elif isinstance(class_idx, int):
            class_idx = torch.full((x.size(0),), class_idx, device=x.device)
        score = logits.gather(1, class_idx.view(-1, 1)).sum()
        score.backward()

        acts = self._as_bchw(self.acts)
        grads = self._as_bchw(self.grads)
        weights = grads.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((weights * acts).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear",
                            align_corners=False)
        return _normalise(cam.squeeze(1).detach())

    def close(self):
        self._h1.remove()
        self._h2.remove()


class GradCAMpp(GradCAM):
    """Grad-CAM++ (Chattopadhay et al., 2018) — better for multiple lesion foci."""

    def __call__(self, x: torch.Tensor, class_idx=None) -> np.ndarray:
        self.model.zero_grad(set_to_none=True)
        logits = self.model(x)
        if class_idx is None:
            class_idx = logits.argmax(dim=1)
        elif isinstance(class_idx, int):
            class_idx = torch.full((x.size(0),), class_idx, device=x.device)
        score = logits.gather(1, class_idx.view(-1, 1)).sum()
        score.backward()

        acts = self._as_bchw(self.acts)
        grads = self._as_bchw(self.grads)
        g2, g3 = grads ** 2, grads ** 3
        denom = 2 * g2 + (acts * g3).sum(dim=(2, 3), keepdim=True)
        alpha = g2 / torch.where(denom != 0, denom, torch.ones_like(denom))
        weights = (alpha * F.relu(grads)).sum(dim=(2, 3), keepdim=True)
        cam = F.relu((weights * acts).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear",
                            align_corners=False)
        return _normalise(cam.squeeze(1).detach())


def fusion_attention_map(model, x: torch.Tensor) -> np.ndarray:
    """Architecture-native explanation for BCAFNet.

    Reads the bidirectional cross-attention weights directly, so the map is what
    the fusion actually computed rather than a gradient approximation of it.
    """
    model.zero_grad(set_to_none=True)
    with torch.no_grad():
        _ = model(x, store_attn=True)
    a_v2c = model.v2c.last_attention   # (B, Nc, Nv)
    a_c2v = model.c2v.last_attention   # (B, Nv, Nc)
    if a_v2c is None or a_c2v is None:
        raise RuntimeError("No attention captured — call with store_attn=True.")

    # How much each conv token is attended to, from both directions.
    imp = a_v2c.mean(dim=2) + a_c2v.mean(dim=1)     # (B, Nc)
    b, n = imp.shape
    s = int(round(n ** 0.5))
    cam = imp.reshape(b, 1, s, s)
    cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear",
                        align_corners=False)
    return _normalise(cam.squeeze(1).detach())


def _normalise(cam: torch.Tensor) -> np.ndarray:
    """Per-sample min-max to [0,1]."""
    b = cam.shape[0]
    flat = cam.reshape(b, -1)
    lo = flat.min(dim=1, keepdim=True).values
    hi = flat.max(dim=1, keepdim=True).values
    flat = (flat - lo) / (hi - lo + 1e-8)
    return flat.reshape(cam.shape).cpu().numpy()


# ============================================================================
# Faithfulness metrics
# ============================================================================
@torch.no_grad()
def deletion_insertion_auc(
    model, x: torch.Tensor, saliency: np.ndarray, class_idx: torch.Tensor,
    n_steps: int = 50, blur_sigma: float = 11.0,
) -> dict:
    """Petsiuk et al. (2018) deletion and insertion curves.

    Deletion removes top-attributed pixels to a blurred baseline; insertion
    reveals them onto that baseline. Reports normalised AUCs and their
    difference, which is the single most informative faithfulness scalar.
    """
    model.eval()
    device = x.device
    b, _, h, w = x.shape
    npix = h * w
    baseline = _gaussian_blur(x, blur_sigma)

    order = torch.tensor(
        saliency.reshape(b, -1).argsort(axis=1)[:, ::-1].copy(), device=device
    )
    step = max(1, npix // n_steps)

    del_scores, ins_scores = [], []
    x_del, x_ins = x.clone(), baseline.clone()
    for k in range(0, npix + step, step):
        p_del = F.softmax(model(x_del), dim=1).gather(1, class_idx.view(-1, 1))
        p_ins = F.softmax(model(x_ins), dim=1).gather(1, class_idx.view(-1, 1))
        del_scores.append(p_del.squeeze(1).cpu().numpy())
        ins_scores.append(p_ins.squeeze(1).cpu().numpy())
        if k >= npix:
            break
        idx = order[:, k:k + step]
        for i in range(b):
            rows, cols = idx[i] // w, idx[i] % w
            x_del[i, :, rows, cols] = baseline[i, :, rows, cols]
            x_ins[i, :, rows, cols] = x[i, :, rows, cols]

    dele = np.stack(del_scores, axis=1)   # (B, T)
    ins = np.stack(ins_scores, axis=1)
    return {
        "deletion_auc": _auc(dele),
        "insertion_auc": _auc(ins),
        "faithfulness_gap": _auc(ins) - _auc(dele),
        "deletion_curve": dele.mean(axis=0),
        "insertion_curve": ins.mean(axis=0),
    }


@torch.no_grad()
def aopc(model, x: torch.Tensor, saliency: np.ndarray,
         class_idx: torch.Tensor, n_steps: int = 20) -> np.ndarray:
    """Area Over the Perturbation Curve (Samek et al., 2017).

    Mean confidence drop as top-attributed regions are progressively removed.
    Higher = the map identified genuinely decision-relevant evidence.
    """
    model.eval()
    b, _, h, w = x.shape
    npix = h * w
    baseline = _gaussian_blur(x, 11.0)
    order = torch.tensor(
        saliency.reshape(b, -1).argsort(axis=1)[:, ::-1].copy(), device=x.device
    )
    p0 = F.softmax(model(x), dim=1).gather(1, class_idx.view(-1, 1)).squeeze(1)
    step = max(1, npix // n_steps)

    drops = []
    xp = x.clone()
    for k in range(0, npix, step):
        idx = order[:, k:k + step]
        for i in range(b):
            rows, cols = idx[i] // w, idx[i] % w
            xp[i, :, rows, cols] = baseline[i, :, rows, cols]
        pk = F.softmax(model(xp), dim=1).gather(1, class_idx.view(-1, 1)).squeeze(1)
        drops.append((p0 - pk).cpu().numpy())
    return np.stack(drops, axis=1).mean(axis=1)


@torch.no_grad()
def adcc(model, x: torch.Tensor, saliency: np.ndarray,
         class_idx: torch.Tensor, cam_fn=None) -> dict:
    """Average Drop / Coherency / Complexity and their harmonic mean (ADCC).

    Poppi et al. (2021). ADCC resists the failure modes of each component:
    a map covering the whole image scores a perfect Average Drop but a terrible
    Complexity, so it cannot win on ADCC.

    cam_fn is REQUIRED for a usable result. Coherency compares the attribution
    of the image against the attribution of the masked image, so the callable
    must regenerate the same attribution method. Without it, coherency and ADCC
    are NaN and the metric is meaningless.
    """
    model.eval()
    b = x.shape[0]
    sal = torch.tensor(saliency, device=x.device, dtype=x.dtype).unsqueeze(1)

    p_full = F.softmax(model(x), dim=1).gather(1, class_idx.view(-1, 1)).squeeze(1)
    p_mask = F.softmax(model(x * sal), dim=1).gather(1, class_idx.view(-1, 1)).squeeze(1)

    avg_drop = torch.clamp(p_full - p_mask, min=0) / (p_full + 1e-8)
    avg_drop = avg_drop.cpu().numpy()

    complexity = sal.reshape(b, -1).abs().mean(dim=1).cpu().numpy()

    if cam_fn is not None:
        # Gradient-based attribution methods need autograd, and this function
        # runs under no_grad. Re-enable it locally, on a detached copy, so the
        # coherency term can actually be computed.
        with torch.enable_grad():
            masked = (x * sal).detach().requires_grad_(True)
            sal2 = cam_fn(masked, class_idx)
        a = saliency.reshape(b, -1)
        c = np.asarray(sal2).reshape(b, -1)
        coherency = np.array([_pearson(a[i], c[i]) for i in range(b)])
        coherency = (coherency + 1.0) / 2.0     # map [-1,1] -> [0,1]
    else:
        import warnings
        warnings.warn(
            "adcc() called without cam_fn: coherency, and therefore ADCC "
            "itself, will be NaN. Pass a callable (x, class_idx) -> saliency "
            "that regenerates the SAME attribution method, or the ADCC column "
            "will be empty in your results table.",
            RuntimeWarning, stacklevel=2,
        )
        coherency = np.full(b, np.nan)

    with np.errstate(invalid="ignore", divide="ignore"):
        terms = np.stack([coherency, 1 - complexity, 1 - avg_drop])
        adcc_val = 3.0 / np.sum(1.0 / np.clip(terms, 1e-8, None), axis=0)

    return {
        "average_drop": avg_drop,
        "coherency": coherency,
        "complexity": complexity,
        "adcc": adcc_val,
    }


# ============================================================================
# helpers
# ============================================================================
def _auc(curve: np.ndarray) -> np.ndarray:
    """Normalised area under a per-sample curve of shape (B, T)."""
    return np.trapezoid(curve, axis=1) / (curve.shape[1] - 1)


def _pearson(a: np.ndarray, b: np.ndarray) -> float:
    a = a - a.mean()
    b = b - b.mean()
    d = np.sqrt((a ** 2).sum() * (b ** 2).sum())
    return float((a * b).sum() / d) if d > 1e-12 else 0.0


def _gaussian_blur(x: torch.Tensor, sigma: float) -> torch.Tensor:
    """Blurred baseline — a more honest reference than zeros, which are
    themselves out of distribution and inflate apparent faithfulness."""
    k = int(2 * round(3 * sigma) + 1)
    coords = torch.arange(k, dtype=x.dtype, device=x.device) - k // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = (g / g.sum()).view(1, 1, 1, -1)
    c = x.shape[1]
    out = F.conv2d(F.pad(x, (k // 2, k // 2, 0, 0), mode="reflect"),
                   g.expand(c, 1, 1, k), groups=c)
    out = F.conv2d(F.pad(out, (0, 0, k // 2, k // 2), mode="reflect"),
                   g.transpose(2, 3).expand(c, 1, k, 1), groups=c)
    return out


def get_target_layer(model, model_key: str):
    """Best-practice Grad-CAM target layer per architecture."""
    if model_key == "bcaf":
        return model.cnn.stages[-1]
    bb = getattr(model, "backbone", model)
    for attr in ("layer4", "features", "stages", "blocks"):
        if hasattr(bb, attr):
            mod = getattr(bb, attr)
            try:
                return mod[-1]
            except (TypeError, IndexError):
                return mod
    raise ValueError(f"Could not resolve a target layer for '{model_key}'.")

Writing /kaggle/working/src/xai.py


In [ ]:
%%writefile /kaggle/working/src/stats.py
"""
Statistical testing for model comparison.

The single most common reviewer objection to BreakHis papers is that a claimed
improvement of a few tenths of a percent is reported without any test of whether
it exceeds run-to-run variance. This module makes every comparison in the paper
defensible: paired tests, multiple-comparison correction, effect sizes, and
confidence intervals — never a bare point estimate.
"""
from __future__ import annotations

import itertools

import numpy as np
import pandas as pd
from scipy import stats


# ----------------------------------------------------------------------------
# Paired tests on the same test set
# ----------------------------------------------------------------------------
def mcnemar_test(correct_a: np.ndarray, correct_b: np.ndarray,
                exact: bool = True) -> dict:
    """McNemar's test on paired per-sample correctness vectors.

    The right test when two models are evaluated on the identical test set:
    it conditions on the discordant pairs and ignores samples both got right or
    both got wrong. Uses the exact binomial form when n is small.
    """
    correct_a = np.asarray(correct_a).astype(bool)
    correct_b = np.asarray(correct_b).astype(bool)
    if correct_a.shape != correct_b.shape:
        raise ValueError("Correctness vectors must be the same length.")

    n01 = int(np.sum(~correct_a & correct_b))   # only B right
    n10 = int(np.sum(correct_a & ~correct_b))   # only A right
    n_disc = n01 + n10

    if n_disc == 0:
        return {"n01": 0, "n10": 0, "statistic": 0.0, "p_value": 1.0,
                "test": "degenerate (no discordant pairs)",
                "odds_ratio": np.nan}

    if exact or n_disc < 25:
        p = float(stats.binomtest(n10, n_disc, 0.5).pvalue)
        statistic, test = float(min(n01, n10)), "exact binomial"
    else:
        statistic = (abs(n10 - n01) - 1) ** 2 / n_disc   # continuity-corrected
        p = float(stats.chi2.sf(statistic, df=1))
        test = "chi-square (continuity corrected)"

    return {"n01": n01, "n10": n10, "statistic": statistic, "p_value": p,
            "test": test,
            "odds_ratio": float(n10 / n01) if n01 > 0 else np.inf}


def corrected_paired_ttest(scores_a: np.ndarray, scores_b: np.ndarray,
                          n_train: int, n_test: int) -> dict:
    """Nadeau & Bengio (2003) corrected resampled t-test.

    A plain paired t-test across cross-validation folds is anti-conservative,
    because the folds share training data and so are not independent. The
    correction inflates the variance by (1/k + n_test/n_train). Using the
    uncorrected test is a common and serious error in this literature.
    """
    a, b = np.asarray(scores_a, float), np.asarray(scores_b, float)
    d = a - b
    k = len(d)
    if k < 2:
        raise ValueError("Need at least 2 folds.")
    mean_d = d.mean()
    var_d = d.var(ddof=1)
    if var_d == 0:
        return {"t_statistic": 0.0, "p_value": 1.0, "mean_difference": float(mean_d),
                "df": k - 1, "correction_factor": np.nan}

    correction = (1.0 / k) + (n_test / n_train)
    t = mean_d / np.sqrt(var_d * correction)
    p = float(2 * stats.t.sf(abs(t), df=k - 1))
    return {"t_statistic": float(t), "p_value": p,
            "mean_difference": float(mean_d), "df": k - 1,
            "correction_factor": float(correction)}


def wilcoxon_test(scores_a: np.ndarray, scores_b: np.ndarray) -> dict:
    """Exact Wilcoxon signed-rank — the non-parametric fold-level comparison.

    Note the floor: with k folds the smallest attainable two-sided p is
    2^-(k-1). At k=5 that is 0.0625, so no result can reach p<0.05. The paper
    must state this rather than reporting 'not significant' as if it were
    evidence of equivalence.
    """
    a, b = np.asarray(scores_a, float), np.asarray(scores_b, float)
    k = len(a)
    min_p = 2.0 ** (-(k - 1))
    if np.allclose(a, b):
        return {"statistic": 0.0, "p_value": 1.0, "n_folds": k,
                "min_attainable_p": min_p, "at_floor": False}
    try:
        res = stats.wilcoxon(a, b, alternative="two-sided", mode="exact")
    except (TypeError, ValueError):
        res = stats.wilcoxon(a, b, alternative="two-sided")
    p = float(res.pvalue)
    return {"statistic": float(res.statistic), "p_value": p, "n_folds": k,
            "min_attainable_p": min_p,
            "at_floor": bool(abs(p - min_p) < 1e-9)}


# ----------------------------------------------------------------------------
# Effect size and intervals
# ----------------------------------------------------------------------------
def cohens_d_paired(a: np.ndarray, b: np.ndarray) -> float:
    """Paired Cohen's d. Report alongside every p-value."""
    d = np.asarray(a, float) - np.asarray(b, float)
    sd = d.std(ddof=1)
    return float(d.mean() / sd) if sd > 0 else 0.0


def bootstrap_ci(values: np.ndarray, statistic=np.mean, n_boot: int = 10000,
                alpha: float = 0.05, seed: int = 42) -> dict:
    """Percentile bootstrap confidence interval."""
    rng = np.random.default_rng(seed)
    v = np.asarray(values, float)
    boots = np.array([statistic(rng.choice(v, size=len(v), replace=True))
                      for _ in range(n_boot)])
    lo, hi = np.percentile(boots, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return {"point": float(statistic(v)), "ci_low": float(lo),
            "ci_high": float(hi), "alpha": alpha}


def bootstrap_metric_ci(y_true: np.ndarray, y_pred: np.ndarray, metric_fn,
                       groups: np.ndarray | None = None,
                       n_boot: int = 2000, alpha: float = 0.05,
                       seed: int = 42) -> dict:
    """Bootstrap CI for any metric.

    If `groups` (patient ids) is supplied, resamples whole PATIENTS rather than
    images. This is the correct unit of resampling: images from one patient are
    not independent, and image-level bootstrapping produces intervals that are
    far too narrow.
    """
    rng = np.random.default_rng(seed)
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    boots = []
    if groups is None:
        n = len(y_true)
        for _ in range(n_boot):
            idx = rng.integers(0, n, n)
            boots.append(metric_fn(y_true[idx], y_pred[idx]))
    else:
        groups = np.asarray(groups)
        uniq = np.unique(groups)
        lookup = {g: np.where(groups == g)[0] for g in uniq}
        for _ in range(n_boot):
            picked = rng.choice(uniq, size=len(uniq), replace=True)
            idx = np.concatenate([lookup[g] for g in picked])
            boots.append(metric_fn(y_true[idx], y_pred[idx]))
    boots = np.asarray(boots, float)
    lo, hi = np.percentile(boots, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return {"point": float(metric_fn(y_true, y_pred)),
            "ci_low": float(lo), "ci_high": float(hi),
            "resample_unit": "patient" if groups is not None else "image"}


# ----------------------------------------------------------------------------
# Multiple comparisons
# ----------------------------------------------------------------------------
def holm_bonferroni(p_values, alpha: float = 0.05) -> pd.DataFrame:
    """Holm-Bonferroni step-down correction.

    Uniformly more powerful than plain Bonferroni at the same family-wise error
    rate. Comparing k models pairwise means k(k-1)/2 tests; without correction
    the chance of at least one spurious 'significant' result is large.
    """
    p = np.asarray(list(p_values), float)
    m = len(p)
    order = np.argsort(p)
    adjusted = np.empty(m)
    running = 0.0
    for rank, i in enumerate(order):
        val = (m - rank) * p[i]
        running = max(running, val)
        adjusted[i] = min(running, 1.0)
    return pd.DataFrame({
        "p_raw": p,
        "p_adjusted": adjusted,
        "reject_null": adjusted < alpha,
    })


def pairwise_model_comparison(fold_scores: dict[str, np.ndarray],
                             n_train: int, n_test: int,
                             alpha: float = 0.05) -> pd.DataFrame:
    """Full pairwise comparison table across models, Holm-corrected.

    fold_scores: {model_name: array of per-fold scores}
    """
    names = list(fold_scores)
    rows = []
    for a, b in itertools.combinations(names, 2):
        sa, sb = np.asarray(fold_scores[a]), np.asarray(fold_scores[b])
        tt = corrected_paired_ttest(sa, sb, n_train, n_test)
        wx = wilcoxon_test(sa, sb)
        rows.append({
            "model_a": a, "model_b": b,
            "mean_a": sa.mean(), "mean_b": sb.mean(),
            "difference": sa.mean() - sb.mean(),
            "cohens_d": cohens_d_paired(sa, sb),
            "p_ttest_corrected": tt["p_value"],
            "p_wilcoxon": wx["p_value"],
            "wilcoxon_at_floor": wx["at_floor"],
        })
    df = pd.DataFrame(rows)
    if len(df):
        holm = holm_bonferroni(df["p_ttest_corrected"], alpha)
        df["p_ttest_holm"] = holm["p_adjusted"].values
        df["significant"] = holm["reject_null"].values
    return df


def equivalence_test(a: np.ndarray, b: np.ndarray, margin: float,
                    alpha: float = 0.05) -> dict:
    """Two one-sided tests (TOST) for statistical equivalence.

    Essential for this paper's central claim. 'p > 0.05' means only that a
    difference was not detected; TOST lets you make the positive claim that any
    difference is smaller than a pre-specified margin. Without it, asserting
    that architectures perform equivalently is an unsupported inference from a
    null result.
    """
    a, b = np.asarray(a, float), np.asarray(b, float)
    d = a - b
    k = len(d)
    se = d.std(ddof=1) / np.sqrt(k)
    if se == 0:
        return {"equivalent": True, "p_lower": 0.0, "p_upper": 0.0,
                "mean_difference": float(d.mean()), "margin": margin}
    t_lo = (d.mean() + margin) / se
    t_hi = (d.mean() - margin) / se
    p_lo = float(stats.t.sf(t_lo, df=k - 1))     # H0: diff <= -margin
    p_hi = float(stats.t.cdf(t_hi, df=k - 1))    # H0: diff >= +margin
    return {"equivalent": bool(max(p_lo, p_hi) < alpha),
            "p_lower": p_lo, "p_upper": p_hi,
            "p_tost": float(max(p_lo, p_hi)),
            "mean_difference": float(d.mean()), "margin": margin}

Writing /kaggle/working/src/stats.py


In [ ]:
%%writefile /kaggle/working/src/figures.py
"""
Publication figure generation.

Schematic figures (study design, architecture, splitting protocol) are drawn
here from code so they are reproducible, vector-based and editable, rather than
redrawn by hand in a slide tool for every revision. IEEE journals want vector
PDF at column width; every figure is emitted as both PDF and 600-dpi PNG.
"""
from __future__ import annotations

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch, Rectangle
import numpy as np

# IEEE single column is 3.5 in; double column 7.16 in.
COL, DCOL = 3.5, 7.16

PALETTE = {
    "cnn":    "#2E6E8E",
    "vit":    "#B5651D",
    "fusion": "#4A7C59",
    "data":   "#5B5B7B",
    "eval":   "#8C4356",
    "light":  "#F2F0EB",
    "ink":    "#22252A",
    "muted":  "#8A8F98",
}


def _setup():
    plt.rcParams.update({
        "font.family": "serif",
        "font.serif": ["DejaVu Serif"],
        "font.size": 8,
        "axes.linewidth": 0.7,
        "savefig.dpi": 600,
        "figure.dpi": 150,
        "pdf.fonttype": 42,      # embed as TrueType — required by IEEE
        "ps.fonttype": 42,
    })


def _save(fig, path_stem: str):
    fig.savefig(f"{path_stem}.pdf", bbox_inches="tight", pad_inches=0.02)
    fig.savefig(f"{path_stem}.png", bbox_inches="tight", pad_inches=0.02, dpi=600)
    plt.close(fig)
    print(f"  wrote {path_stem}.pdf / .png")


def _box(ax, x, y, w, h, label, color, fontsize=7, text_color="white", alpha=1.0):
    ax.add_patch(FancyBboxPatch(
        (x, y), w, h, boxstyle="round,pad=0.012,rounding_size=0.02",
        linewidth=0.8, edgecolor=color, facecolor=color, alpha=alpha, zorder=2))
    ax.text(x + w / 2, y + h / 2, label, ha="center", va="center",
            fontsize=fontsize, color=text_color, zorder=3, linespacing=1.35)


def _arrow(ax, p0, p1, color=None, style="-|>", lw=0.9, rad=0.0):
    ax.add_patch(FancyArrowPatch(
        p0, p1, arrowstyle=style, mutation_scale=8, linewidth=lw,
        color=color or PALETTE["ink"], zorder=1,
        connectionstyle=f"arc3,rad={rad}"))


# ---------------------------------------------------------------------------
# Figure 1 — study design
# ---------------------------------------------------------------------------
def fig_study_design(out="figures/fig1_study_design"):
    _setup()
    fig, ax = plt.subplots(figsize=(DCOL, 3.0))
    ax.set_xlim(0, 10); ax.set_ylim(0, 4.6); ax.axis("off")

    _box(ax, 0.05, 3.4, 1.7, 0.85,
         "BreakHis\n7,909 images\n82 patients\n4 magnifications",
         PALETTE["data"], 6.5)
    _box(ax, 0.05, 2.2, 1.7, 0.85,
         "Integrity check\nvs published\ncounts", PALETTE["data"], 6.5, alpha=0.78)

    _box(ax, 2.15, 3.4, 2.0, 0.85,
         "Patient-wise\nStratifiedGroupKFold\n(k = 5)", PALETTE["eval"], 6.5)
    _box(ax, 2.15, 2.2, 2.0, 0.85,
         "Image-wise split\n(leaky control)", PALETTE["muted"], 6.5)

    _box(ax, 4.55, 3.4, 1.9, 0.85,
         "BCAF hybrid\n+ ablation\n+ 4 baselines", PALETTE["fusion"], 6.5)
    _box(ax, 4.55, 2.2, 1.9, 0.85,
         "Identical training\nrecipe for all", PALETTE["fusion"], 6.5, alpha=0.72)

    for i, (lbl, yy) in enumerate([
        ("Discrimination\nacc / F1 / AUC / MCC", 3.55),
        ("Calibration\nECE / ACE / Brier", 2.60),
        ("Faithfulness\ndel-ins / AOPC / ADCC", 1.65),
    ]):
        _box(ax, 6.85, yy, 1.75, 0.72, lbl, PALETTE["cnn"], 6.3,
             alpha=1 - 0.12 * i)

    _box(ax, 8.95, 2.60, 1.0, 0.72,
         "Statistical\ntesting\n+ TOST", PALETTE["vit"], 6.3)

    _arrow(ax, (1.75, 3.82), (2.15, 3.82))
    _arrow(ax, (1.75, 2.62), (2.15, 2.62))
    _arrow(ax, (0.9, 3.4), (0.9, 3.05))
    _arrow(ax, (4.15, 3.82), (4.55, 3.82))
    _arrow(ax, (4.15, 2.62), (4.55, 2.62))
    for yy in (3.91, 2.96, 2.01):
        _arrow(ax, (6.45, 3.6), (6.85, yy + 0.36), rad=0.12)
    for yy in (3.91, 2.96, 2.01):
        _arrow(ax, (8.6, yy), (8.95, 2.96), rad=-0.1)

    ax.text(0.05, 1.55, "Leakage guard: every fold assignment is re-verified before each fit;\n"
                        "a patient appearing in two folds raises rather than degrading silently.",
            fontsize=6.2, color=PALETTE["ink"], style="italic", va="top")
    ax.text(0.05, 0.75, "Novel relative to prior BreakHis work: calibration reporting, "
                        "quantified explanation faithfulness with a random-saliency control,\n"
                        "equivalence testing, and a like-for-like measurement of the "
                        "leakage-induced inflation.",
            fontsize=6.2, color=PALETTE["eval"], va="top")
    _save(fig, out)


# ---------------------------------------------------------------------------
# Figure 3 — BCAF architecture
# ---------------------------------------------------------------------------
def fig_architecture(out="figures/fig3_architecture"):
    _setup()
    fig, ax = plt.subplots(figsize=(DCOL, 2.9))
    ax.set_xlim(0, 10.4); ax.set_ylim(0, 4.2); ax.axis("off")

    _box(ax, 0.05, 1.75, 0.95, 0.75, "H&E\npatch\n224²", PALETTE["data"], 6.4)

    _box(ax, 1.30, 2.75, 1.78, 0.72, "ConvNeXt-T\nlocal (cytology)", PALETTE["cnn"], 6.4)
    _box(ax, 1.30, 0.85, 1.78, 0.72, "Swin-T\nglobal (architecture)", PALETTE["vit"], 6.4)

    _box(ax, 3.25, 2.75, 0.85, 0.72, "linear\nproj", PALETTE["cnn"], 6.2, alpha=0.75)
    _box(ax, 3.25, 0.85, 0.85, 0.72, "linear\nproj", PALETTE["vit"], 6.2, alpha=0.75)

    _box(ax, 4.45, 2.75, 1.75, 0.72,
         "cross-attn\nQ: conv  KV: swin", PALETTE["fusion"], 6.2)
    _box(ax, 4.45, 0.85, 1.75, 0.72,
         "cross-attn\nQ: swin  KV: conv", PALETTE["fusion"], 6.2)

    _box(ax, 6.6, 1.75, 1.15, 0.85, "gated\nsum\nw ∈ ℝ²", PALETTE["fusion"], 6.4, alpha=0.9)
    _box(ax, 8.15, 1.75, 1.0, 0.85, "LN +\nlinear", PALETTE["ink"], 6.4)
    _box(ax, 9.5, 1.9, 0.8, 0.55, "logits", PALETTE["eval"], 6.4)

    _arrow(ax, (1.0, 2.3), (1.30, 3.11), rad=0.15)
    _arrow(ax, (1.0, 1.95), (1.30, 1.21), rad=-0.15)
    _arrow(ax, (3.08, 3.11), (3.25, 3.11))
    _arrow(ax, (3.08, 1.21), (3.25, 1.21))
    _arrow(ax, (4.1, 3.11), (4.45, 3.11))
    _arrow(ax, (4.1, 1.21), (4.45, 1.21))
    # the bidirectional exchange
    _arrow(ax, (4.35, 2.85), (4.35, 1.5), color=PALETTE["muted"], rad=-0.35, style="<|-|>")
    _arrow(ax, (6.2, 3.11), (6.6, 2.4), rad=0.12)
    _arrow(ax, (6.2, 1.21), (6.6, 1.95), rad=-0.12)
    _arrow(ax, (7.75, 2.17), (8.15, 2.17))
    _arrow(ax, (9.15, 2.17), (9.5, 2.17))

    ax.text(3.6, 2.15, "bidirectional\nexchange", fontsize=5.8,
            color=PALETTE["muted"], ha="center", style="italic")
    ax.text(0.05, 0.35,
            "The gate weights w are recorded per sample, so the paper can report "
            "how much each stream actually contributed —\nevidence about what the "
            "fusion does, not merely that it exists. Cross-attention maps double as "
            "architecture-native explanations.",
            fontsize=6.1, color=PALETTE["ink"], va="top")
    _save(fig, out)


# ---------------------------------------------------------------------------
# Figure 4 — why patient-wise splitting matters
# ---------------------------------------------------------------------------
def fig_splitting(out="figures/fig4_splitting"):
    _setup()
    fig, axes = plt.subplots(1, 2, figsize=(DCOL, 2.5))
    rng = np.random.default_rng(3)
    n_pat, per = 12, 6

    for ax, mode in zip(axes, ["image-wise (leaky)", "patient-wise (correct)"]):
        ax.set_xlim(-0.5, per + 0.4); ax.set_ylim(-2.4, n_pat + 0.2)
        ax.axis("off")
        for p in range(n_pat):
            if mode.startswith("image"):
                assign = rng.integers(0, 2, per)          # split within a patient
            else:
                assign = np.full(per, 0 if p < 8 else 1)  # whole patient moves
            for i, a in enumerate(assign):
                ax.add_patch(Rectangle(
                    (i, n_pat - p - 1), 0.86, 0.72,
                    facecolor=PALETTE["cnn"] if a == 0 else PALETTE["eval"],
                    edgecolor="white", linewidth=0.5))
            ax.text(-0.28, n_pat - p - 0.64, f"P{p+1}", fontsize=5.2,
                    ha="right", va="center", color=PALETTE["ink"])
        ax.set_title(mode, fontsize=8, pad=6)
        if mode.startswith("image"):
            ax.text(per / 2, -1.35,
                    "same patient in train AND test\n→ inflated accuracy",
                    fontsize=6, ha="center", va="center", color=PALETTE["eval"])
        else:
            ax.text(per / 2, -1.35,
                    "each patient in exactly one split\n→ generalisation to unseen patients",
                    fontsize=6, ha="center", va="center", color=PALETTE["fusion"])

    from matplotlib.lines import Line2D
    fig.legend(handles=[
        Line2D([0], [0], marker="s", color="w", label="train",
               markerfacecolor=PALETTE["cnn"], markersize=6),
        Line2D([0], [0], marker="s", color="w", label="test",
               markerfacecolor=PALETTE["eval"], markersize=6)],
        loc="lower center", ncol=2, frameon=False, fontsize=6.5,
        bbox_to_anchor=(0.5, -0.02))
    fig.text(0.5, -0.09, "Rows are patients; cells are images from that patient.",
             ha="center", fontsize=6, style="italic", color=PALETTE["muted"])
    _save(fig, out)


def build_all(out_dir="figures"):
    import os
    os.makedirs(out_dir, exist_ok=True)
    print("Generating schematic figures:")
    fig_study_design(f"{out_dir}/fig1_study_design")
    fig_architecture(f"{out_dir}/fig3_architecture")
    fig_splitting(f"{out_dir}/fig4_splitting")


if __name__ == "__main__":
    import sys
    build_all(sys.argv[1] if len(sys.argv) > 1 else "figures")

Writing /kaggle/working/src/figures.py


In [ ]:
# Confirm every module imports cleanly before anything expensive runs
import importlib, subprocess
try:
    import timm
except ImportError:
    subprocess.run([sys.executable,"-m","pip","install","-q","timm"], check=True)

import torch
for m in ["data","models","engine","calibration","xai","stats","figures"]:
    importlib.import_module(m)
print("all modules import OK")
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
if not torch.cuda.is_available():
    print("\nWARNING: no GPU detected. Enable it under Settings -> Accelerator "
          "before running the training cells, or this will take days.")

all modules import OK
torch 2.11.0+cu128 | CUDA True | Tesla T4


## 4. Configuration and run plan

`RUN_PLAN` is ordered by scientific value, not by convenience. The two runs the reviewer
marked CRITICAL — the concatenation-fusion ablation and Swin-Tiny alone at 40x — come first,
so that even a one-hour session delivers them.

In [ ]:
import numpy as np, pandas as pd, time, torch
from dataclasses import asdict
import data, models, engine, calibration as cal, stats as S, xai

# ---------------------------------------------------------------- dataset root
# Resolve BreaKHis WITHOUT searching the mounted Google Drive. A Drive mount
# lives under /content, the generic search path, and typically holds tens of
# thousands of unrelated images -- which the search will happily find instead.
DATA_ROOT_OVERRIDE = ""     # set if you already have a copy, e.g. "/content/breakhis"

def _usable(root):
    try:
        idx = data.build_index(root)
    except Exception:
        return None
    return root if len(idx) >= 7000 else None

def resolve_breakhis():
    if DATA_ROOT_OVERRIDE:
        r = _usable(DATA_ROOT_OVERRIDE)
        if r:
            return r
        raise RuntimeError(
            f"DATA_ROOT_OVERRIDE={DATA_ROOT_OVERRIDE} does not contain BreaKHis.")

    # 1. an existing copy, in places that are never the Drive mount
    for c in ["/kaggle/input", "/content/breakhis", "/content/BreaKHis_v1",
              os.path.expanduser("~/.cache/kagglehub"), "/root/.cache/kagglehub",
              "./BreaKHis_v1", "./data"]:
        if os.path.isdir(c):
            r = _usable(c)
            if r:
                print("using existing copy at", c)
                return r

    # 2. otherwise download it (about 4 GB; a few minutes on Colab)
    print("downloading BreaKHis via kagglehub ...")
    root = str(data.acquire_via_kagglehub())
    r = _usable(root)
    if r:
        return r
    raise RuntimeError(
        f"kagglehub returned {root} but no BreaKHis images were found there. "
        "Set DATA_ROOT_OVERRIDE to a folder that preserves the original "
        "BreaKHis_v1 filenames.")

DATA_ROOT = resolve_breakhis()
print("dataset root:", DATA_ROOT)

_probe = data.build_index(DATA_ROOT)
print(f"index: {len(_probe)} images, {_probe.patient_id.nunique()} patients")
assert len(_probe) == 7909, (
    f"expected 7909 images, found {len(_probe)}. This mirror is incomplete or "
    "renamed; a patient-wise study cannot be run on it.")
del _probe

CFG = engine.TrainConfig(
    num_classes=2, task="binary", img_size=224,
    batch_size=32, epochs=30, lr=3e-4, backbone_lr_mult=0.1,
    weight_decay=0.05, patience=8, amp=True, num_workers=2,
    seed=42, protocol="patient_wise",
)
N_FOLDS   = 5
LABEL_COL = "binary_label"
DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MAGS   = [40, 100, 200, 400]
FAMILY = ["bcaf", "concat_ablation", "swin_tiny", "convnext_tiny", "resnet50"]

# (magnification, model) in the order they will run.
RUN_PLAN = (
    # --- CRITICAL: the ablation and the missing half of the hybrid, at 40x
    [(40, "concat_ablation"), (40, "swin_tiny"), (40, "resnet50")]
    # --- the two models already reported at 40x (skipped if adopted in Section 4b)
    + [(40, "bcaf"), (40, "convnext_tiny")]
    # --- HIGH: the rest of the magnification sweep
    + [(m, k) for m in (100, 200, 400) for k in FAMILY]
)

LEAK_MAGS  = [40, 100, 200, 400]   # image-wise control arm
LEAK_MODEL = "bcaf"
XAI_MAG, XAI_N, XAI_STEPS = 40, 200, 50
XAI_FOLDS  = [0, 1, 2, 3, 4]       # the review asks for all five, not just fold 0

json.dump(asdict(CFG), open(f"{OUT}/config.json", "w"), indent=2)
print(f"{len(RUN_PLAN)} cross-validation runs planned; "
      f"~30 min each on an A100 => roughly {len(RUN_PLAN)*0.5:.0f} GPU-hours.")
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
if not torch.cuda.is_available():
    print("\nSTOP: no GPU. Runtime -> Change runtime type -> GPU.")

downloading BreaKHis via kagglehub ...
No local copy found. Downloading 'ambarish/breakhis' via kagglehub (a few minutes, ~4 GB)...
Using Colab cache for faster access to the 'breakhis' dataset.
Downloaded to: /kaggle/input/breakhis
dataset root: /kaggle/input/breakhis
index: 7909 images, 81 patients
20 cross-validation runs planned; ~30 min each on an A100 => roughly 10 GPU-hours.
torch 2.11.0+cu128 | CUDA True | Tesla T4


### 4b. Adopt the 40x results you already have (optional but recommended)

If you still have the results zip from the completed run, put it in Drive and give its path
below. The two 40x models already reported in the paper are then reused rather than retrained,
which saves about an hour and — more importantly — keeps the numbers in the manuscript
identical to the numbers in this sweep.

In [ ]:
PRIOR_ZIP = ""          # e.g. "/content/drive/MyDrive/bcaf_results.zip"; leave "" to skip
PRIOR_DIR = ""          # or an already-unzipped folder

import zipfile, glob

def adopt_prior_40x(zip_path="", src_dir=""):
    # Copy previously computed 40x results into the sweep's naming scheme.
    staging = f"{OUT}/_prior"
    if zip_path and os.path.exists(zip_path):
        os.makedirs(staging, exist_ok=True)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(staging)
        src_dir = staging
    if not src_dir or not os.path.isdir(src_dir):
        print("No prior results supplied — the 40x models will be trained from scratch.")
        return 0
    n = 0
    for key in ("bcaf", "convnext_tiny"):
        for stem in ("fold_metrics", "oof"):
            hits = glob.glob(f"{src_dir}/**/{stem}_{key}.csv", recursive=True)
            if not hits:
                continue
            dst = f"{OUT}/{stem}_{key}_m40.csv"
            if not os.path.exists(dst):
                shutil.copy(hits[0], dst)
                n += 1
                print("adopted", os.path.basename(hits[0]), "->", os.path.basename(dst))
    # checkpoints, if the zip carried them, let Section 8 skip retraining
    for cp in glob.glob(f"{src_dir}/**/*_fold*.pt", recursive=True):
        dst = f"{CKPT}/{os.path.basename(cp).replace('.pt', '_m40.pt')}"
        if not os.path.exists(dst):
            shutil.copy(cp, dst); n += 1
    print(f"adopted {n} prior files")
    return n

adopt_prior_40x(PRIOR_ZIP, PRIOR_DIR)

No prior results supplied — the 40x models will be trained from scratch.


0

## 5. Dataset integrity check

The root was resolved and the image count verified in Section 4. This section records the per-subtype and per-magnification composition.

In [ ]:
df_all = data.build_index(DATA_ROOT)
print(f"indexed {len(df_all)} images, {df_all.patient_id.nunique()} patients")

ver = data.verify_against_published(df_all)
ver.to_csv(f"{OUT}/dataset_verification.csv", index=False)
display(ver)

mag_tbl = (df_all.groupby(["magnification", "binary_label"]).size()
           .unstack(fill_value=0).rename(columns={0: "benign", 1: "malignant"}))
mag_tbl["total"] = mag_tbl.sum(axis=1)
mag_tbl["pct_malignant"] = (mag_tbl.malignant / mag_tbl.total * 100).round(2)
mag_tbl.to_csv(f"{OUT}/magnification_distribution.csv")
display(mag_tbl)

indexed 7909 images, 81 patients


,class,subtype,published,found,delta,match
0,benign,adenosis,444,444,0,True
1,benign,fibroadenoma,1014,1014,0,True
2,benign,phyllodes_tumor,453,453,0,True
3,benign,tubular_adenoma,569,569,0,True
4,malignant,ductal_carcinoma,3451,3451,0,True
5,malignant,lobular_carcinoma,626,626,0,True
6,malignant,mucinous_carcinoma,792,792,0,True
7,malignant,papillary_carcinoma,560,560,0,True
8,TOTAL,,7909,7909,0,True


binary_label,benign,malignant,total,pct_malignant
magnification,,,,
40,625,1370,1995,68.67
100,644,1437,2081,69.05
200,623,1390,2013,69.05
400,588,1232,1820,67.69


## 6. The sweep

Each `(magnification, model)` pair writes two files as soon as it finishes. On restart,
pairs whose files exist are skipped, so nothing is ever recomputed.

**This is the long cell.** It prints a running estimate of time remaining. You can interrupt
it at any point and re-run it later.

In [ ]:
FOLD_INDEX = {}     # magnification -> dataframe with fold assignments

def folded(mag):
    # Patient-wise folds for one magnification, built once and cached.
    if mag in FOLD_INDEX:
        return FOLD_INDEX[mag]
    d = df_all[df_all.magnification == int(mag)].copy()
    if d.empty:
        raise ValueError(f"no images at {mag}x — check the index")
    d = data.make_patient_wise_folds(d, n_splits=N_FOLDS, label_col=LABEL_COL, seed=CFG.seed)
    data.assert_no_patient_leakage(d)
    comp = data.split_summary(d)
    comp["magnification"] = mag
    comp.to_csv(f"{OUT}/fold_composition_m{mag}.csv", index=False)
    print(f"[{mag}x] {len(d)} images, {d.patient_id.nunique()} grouping units, "
          f"folds verified disjoint")
    FOLD_INDEX[mag] = d
    return d

def build_for(key):
    return lambda: models.build_model(key, CFG.num_classes, True, CFG.img_size)

def run_one(mag, key):
    fold_csv = f"{OUT}/fold_metrics_{key}_m{mag}.csv"
    oof_csv  = f"{OUT}/oof_{key}_m{mag}.csv"
    if os.path.exists(fold_csv) and os.path.exists(oof_csv):
        print(f"[{mag}x {key}] already done — skipping")
        return False
    d = folded(mag)
    print(f"\n{'='*72}\n  {mag}x  |  {key}\n{'='*72}")
    cfg = engine.TrainConfig(**{**asdict(CFG), "model_key": key,
                                "magnification": str(mag)})
    t0 = time.time()
    res = engine.run_cross_validation(
        d, cfg, build_fn=build_for(key), device=DEVICE,
        label_col=LABEL_COL, verbose=True,
        ckpt_dir=CKPT if (mag == XAI_MAG) else None)
    fold_df, oof = res[0], res[1]          # a third element appears only with ckpt_dir
    mins = round((time.time() - t0) / 60, 1)
    fold_df["model"] = key; fold_df["magnification"] = mag; fold_df["minutes"] = mins
    oof["model"] = key;     oof["magnification"] = mag
    fold_df.to_csv(fold_csv, index=False)
    oof.to_csv(oof_csv, index=False)
    print(f"[{mag}x {key}] done in {mins} min -> {os.path.basename(fold_csv)}")
    return True

# Rename 40x checkpoints written by run_one so Section 9 finds them
def _tag_ckpts(mag):
    for p in glob.glob(f"{CKPT}/*_fold*.pt"):
        if f"_m{mag}" not in p:
            os.rename(p, p.replace(".pt", f"_m{mag}.pt"))

done_t, todo = [], [p for p in RUN_PLAN]
for i, (mag, key) in enumerate(RUN_PLAN, 1):
    t0 = time.time()
    ran = run_one(mag, key)
    if ran:
        done_t.append(time.time() - t0)
        if mag == XAI_MAG:
            _tag_ckpts(mag)
        if done_t:
            avg = sum(done_t) / len(done_t)
            left = (len(RUN_PLAN) - i) * avg / 3600
            print(f"    progress {i}/{len(RUN_PLAN)} | ~{left:.1f} GPU-hours remaining")
print("\nSWEEP COMPLETE")

[40x concat_ablation] already done — skipping
[40x swin_tiny] already done — skipping
[40x resnet50] already done — skipping
[40x bcaf] already done — skipping
[40x convnext_tiny] already done — skipping
[100x bcaf] already done — skipping
[100x concat_ablation] already done — skipping
[100x swin_tiny] already done — skipping
[100x convnext_tiny] already done — skipping
[100x resnet50] already done — skipping
[200x bcaf] already done — skipping
[200x concat_ablation] already done — skipping
[200x swin_tiny] already done — skipping
[200x convnext_tiny] already done — skipping
[200x resnet50] already done — skipping
[400x bcaf] already done — skipping
[400x concat_ablation] already done — skipping
[400x swin_tiny] already done — skipping
[400x convnext_tiny] already done — skipping
[400x resnet50] already done — skipping

SWEEP COMPLETE


## 7. Image-wise leakage arm at every magnification

In [ ]:
leak_rows = []
for mag in LEAK_MAGS:
    path = f"{OUT}/leakage_m{mag}.csv"
    if os.path.exists(path):
        leak_rows.append(pd.read_csv(path)); print(f"[{mag}x] leakage arm cached"); continue

    clean_csv = f"{OUT}/fold_metrics_{LEAK_MODEL}_m{mag}.csv"
    if not os.path.exists(clean_csv):
        print(f"[{mag}x] no patient-wise result for {LEAK_MODEL} yet — run Section 6 first")
        continue
    clean = pd.read_csv(clean_csv)

    d = df_all[df_all.magnification == int(mag)].copy()
    d_leaky = data.make_image_wise_folds(d, n_splits=N_FOLDS,
                                         label_col=LABEL_COL, seed=CFG.seed)
    n_shared = int(d_leaky.groupby("patient_id")["fold"].nunique().gt(1).sum())
    print(f"[{mag}x] image-wise splitting puts {n_shared} of "
          f"{d_leaky.patient_id.nunique()} patients into multiple folds")

    cfg_leaky = engine.TrainConfig(**{**asdict(CFG), "model_key": LEAK_MODEL,
                                      "protocol": "image_wise", "magnification": str(mag)})
    fd, _ = engine.run_cross_validation(
        d_leaky, cfg_leaky, build_fn=build_for(LEAK_MODEL), device=DEVICE,
        label_col=LABEL_COL, verbose=True)

    row = pd.DataFrame([
        {"magnification": mag, "protocol": "patient-wise (leakage-free)",
         "accuracy": clean.img_accuracy.mean(),
         "accuracy_std": clean.img_accuracy.std(ddof=1),
         "balanced_accuracy": clean.img_balanced_accuracy.mean(),
         "auc": clean.img_auc.mean(), "patients_shared_across_folds": 0},
        {"magnification": mag, "protocol": "image-wise (leaky)",
         "accuracy": fd.img_accuracy.mean(),
         "accuracy_std": fd.img_accuracy.std(ddof=1),
         "balanced_accuracy": fd.img_balanced_accuracy.mean(),
         "auc": fd.img_auc.mean(), "patients_shared_across_folds": n_shared}])
    row["inflation_pp"] = (row.accuracy - row.accuracy.iloc[0]) * 100
    row["sd_ratio"] = row.accuracy_std.iloc[0] / row.accuracy_std
    row.to_csv(path, index=False)
    leak_rows.append(row)

if leak_rows:
    leak_all = pd.concat(leak_rows, ignore_index=True)
    leak_all.to_csv(f"{OUT}/leakage_by_magnification.csv", index=False)
    display(leak_all.round(4))

[40x] image-wise splitting puts 80 of 81 patients into multiple folds

--- fold 0: train 1596 imgs / 81 pts | val 399 imgs / 78 pts ---


/kaggle/working/src/engine.py:246: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  sched.step()


  ep 00 loss 0.5604 acc 0.9173 bal 0.9180 auc 0.9760 (22.5s)  <- best
  ep 01 loss 0.3056 acc 0.9499 bal 0.9526 auc 0.9930 (20.6s)  <- best
  ep 02 loss 0.3564 acc 0.9373 bal 0.9174 auc 0.9911 (20.5s)
  ep 03 loss 0.2491 acc 0.9749 bal 0.9600 auc 0.9995 (20.5s)  <- best
  ep 04 loss 0.2512 acc 0.9098 bal 0.8560 auc 0.9966 (20.4s)
  ep 05 loss 0.2868 acc 0.9549 bal 0.9280 auc 0.9994 (20.6s)
  ep 06 loss 0.1948 acc 0.9950 bal 0.9942 auc 0.9999 (20.5s)  <- best
  ep 07 loss 0.1970 acc 0.9699 bal 0.9520 auc 0.9992 (20.5s)
  ep 08 loss 0.1769 acc 0.9799 bal 0.9680 auc 0.9994 (20.4s)
  ep 09 loss 0.1926 acc 0.9900 bal 0.9840 auc 0.9997 (20.6s)
  ep 10 loss 0.1580 acc 0.9950 bal 0.9920 auc 0.9999 (20.5s)
  ep 11 loss 0.1565 acc 0.9950 bal 0.9920 auc 0.9997 (20.5s)
  ep 12 loss 0.1673 acc 0.9950 bal 0.9964 auc 1.0000 (20.5s)  <- best
  ep 13 loss 0.1446 acc 0.9975 bal 0.9960 auc 1.0000 (20.5s)
  ep 14 loss 0.1474 acc 0.9975 bal 0.9960 auc 1.0000 (20.4s)
  ep 15 loss 0.1540 acc 0.9925 bal 0.990

/kaggle/working/src/engine.py:246: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  sched.step()


  ep 00 loss 0.5310 acc 0.8922 bal 0.9041 auc 0.9763 (20.5s)  <- best
  ep 01 loss 0.3283 acc 0.9424 bal 0.9363 auc 0.9872 (20.6s)  <- best
  ep 02 loss 0.2832 acc 0.9348 bal 0.8982 auc 0.9929 (20.5s)
  ep 03 loss 0.3416 acc 0.9474 bal 0.9421 auc 0.9928 (20.5s)  <- best
  ep 04 loss 0.2581 acc 0.9850 bal 0.9760 auc 0.9998 (20.5s)  <- best
  ep 05 loss 0.2278 acc 0.9925 bal 0.9902 auc 0.9980 (20.5s)  <- best
  ep 06 loss 0.2075 acc 0.9799 bal 0.9680 auc 0.9994 (20.6s)
  ep 07 loss 0.2190 acc 0.9699 bal 0.9542 auc 0.9964 (20.6s)
  ep 08 loss 0.1955 acc 0.9699 bal 0.9520 auc 0.9994 (20.5s)
  ep 09 loss 0.2011 acc 0.9273 bal 0.9471 auc 0.9981 (20.6s)
  ep 10 loss 0.1660 acc 0.9850 bal 0.9782 auc 0.9961 (20.6s)
  ep 11 loss 0.1879 acc 0.9875 bal 0.9844 auc 0.9987 (20.4s)
  ep 12 loss 0.1470 acc 0.9850 bal 0.9804 auc 0.9987 (20.6s)
  ep 13 loss 0.1554 acc 0.9875 bal 0.9844 auc 0.9988 (20.6s)
  early stop at epoch 13 (best epoch 5, bal 0.9902)

--- fold 2: train 1596 imgs / 80 pts | val 399 i

/kaggle/working/src/engine.py:246: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  sched.step()


... (32 outputs left out)

  ep 00 loss 0.5865 acc 0.8791 bal 0.8665 auc 0.9294 (18.8s)  <- best
  ep 01 loss 0.3906 acc 0.8709 bal 0.8979 auc 0.9850 (18.8s)  <- best
  ep 02 loss 0.4862 acc 0.9148 bal 0.8753 auc 0.9846 (19.0s)
  ep 03 loss 0.3475 acc 0.9615 bal 0.9561 auc 0.9907 (18.7s)  <- best
  ep 04 loss 0.2764 acc 0.9478 bal 0.9526 auc 0.9928 (18.9s)
  ep 05 loss 0.2731 acc 0.9258 bal 0.9363 auc 0.9867 (18.7s)
  ep 06 loss 0.2508 acc 0.9478 bal 0.9349 auc 0.9872 (18.8s)
  ep 07 loss 0.2300 acc 0.9286 bal 0.9427 auc 0.9903 (18.7s)
  ep 08 loss 0.2340 acc 0.9451 bal 0.9505 auc 0.9901 (18.7s)
  ep 09 loss 0.2010 acc 0.9368 bal 0.9400 auc 0.9719 (19.0s)
  ep 10 loss 0.2432 acc 0.9396 bal 0.9509 auc 0.9906 (19.1s)
  ep 11 loss 0.1845 acc 0.9725 bal 0.9620 auc 0.9942 (18.8s)  <- best
  ep 12 loss 0.1684 acc 0.9725 bal 0.9687 auc 0.9949 (18.8s)  <- best
  ep 13 loss 0.1528 acc 0.9835 bal 0.9768 auc 0.9960 (18.7s)  <- best
  ep 14 loss 0.1716 acc 0.9478 bal 0.9570 auc 0.9900 (18.7s)
  ep 15 loss 0.1711 acc 0.9725 

/kaggle/working/src/engine.py:246: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  sched.step()


  ep 00 loss 0.5938 acc 0.8654 bal 0.8321 auc 0.9106 (18.8s)  <- best
  ep 01 loss 0.4344 acc 0.8681 bal 0.8870 auc 0.9534 (18.9s)  <- best
  ep 02 loss 0.3657 acc 0.9231 bal 0.9188 auc 0.9733 (19.1s)  <- best
  ep 03 loss 0.3256 acc 0.9231 bal 0.9343 auc 0.9831 (18.8s)  <- best
  ep 04 loss 0.2903 acc 0.8874 bal 0.9145 auc 0.9893 (18.7s)
  ep 05 loss 0.2442 acc 0.9533 bal 0.9566 auc 0.9910 (18.8s)  <- best
  ep 06 loss 0.2100 acc 0.9341 bal 0.9424 auc 0.9894 (18.8s)
  ep 07 loss 0.2450 acc 0.9478 bal 0.9459 auc 0.9893 (18.8s)
  ep 08 loss 0.2483 acc 0.9560 bal 0.9587 auc 0.9938 (18.8s)  <- best
  ep 09 loss 0.2294 acc 0.9615 bal 0.9451 auc 0.9952 (18.8s)
  ep 10 loss 0.1848 acc 0.9670 bal 0.9514 auc 0.9940 (18.8s)
  ep 11 loss 0.1697 acc 0.9560 bal 0.9609 auc 0.9958 (18.8s)  <- best
  ep 12 loss 0.1813 acc 0.9615 bal 0.9517 auc 0.9949 (18.9s)
  ep 13 loss 0.1925 acc 0.9753 bal 0.9729 auc 0.9961 (18.9s)  <- best
  ep 14 loss 0.1687 acc 0.9615 bal 0.9649 auc 0.9964 (19.2s)
  ep 15 loss 

,magnification,protocol,accuracy,accuracy_std,balanced_accuracy,auc,patients_shared_across_folds,inflation_pp,sd_ratio
0,40,patient-wise (leakage-free),0.9123,0.0608,0.9111,0.9601,0,0.0000,1.0000
1,40,image-wise (leaky),0.9920,0.0060,0.9924,0.9983,80,7.9645,10.1677
2,100,patient-wise (leakage-free),0.9209,0.0228,0.9327,0.9801,0,0.0000,1.0000
3,100,image-wise (leaky),0.9889,0.0047,0.9894,0.9981,81,6.8078,4.8677
4,200,patient-wise (leakage-free),0.8951,0.0606,0.8971,0.9422,0,0.0000,1.0000
5,200,image-wise (leaky),0.9920,0.0057,0.9903,0.9982,81,9.6995,10.6754
6,400,patient-wise (leakage-free),0.8763,0.0525,0.8598,0.9166,0,0.0000,1.0000
7,400,image-wise (leaky),0.9852,0.0094,0.9855,0.9972,81,10.8824,5.5680


## 8. Calibration for every model and magnification

In [ ]:
import torch.nn.functional as Fnn

cal_rows = []
for mag in MAGS:
    for key in FAMILY:
        p = f"{OUT}/oof_{key}_m{mag}.csv"
        if not os.path.exists(p):
            continue
        o = pd.read_csv(p)
        lcols = [c for c in o.columns if c.startswith("logit_")]
        pcols = [c for c in o.columns if c.startswith("prob_")]
        y = o["y_true"].to_numpy()
        folds = o["fold"].to_numpy()

        # fold-honest: temperature fitted on the other folds, applied to the held-out one
        for f in sorted(np.unique(folds)):
            fit_m, app_m = folds != f, folds == f
            ts = cal.TemperatureScaler().fit(
                torch.tensor(o.loc[fit_m, lcols].values).float(),
                torch.tensor(o.loc[fit_m, "y_true"].values))
            lg = torch.tensor(o.loc[app_m, lcols].values).float()
            p_after  = Fnn.softmax(ts(lg), dim=1).detach().numpy()
            p_before = o.loc[app_m, pcols].to_numpy()
            cal_rows.append({
                "magnification": mag, "model": key, "fold": int(f),
                "n": int(app_m.sum()), "T": float(ts.temperature),
                "ece_before": cal.expected_calibration_error(p_before, y[app_m]),
                "ece_after":  cal.expected_calibration_error(p_after,  y[app_m]),
                "brier_before": cal.brier_score(p_before, y[app_m]),
                "brier_after":  cal.brier_score(p_after,  y[app_m]),
            })

if cal_rows:
    calib = pd.DataFrame(cal_rows)
    calib.to_csv(f"{OUT}/calibration_per_fold.csv", index=False)
    summ = (calib.groupby(["magnification", "model"])
            [["T", "ece_before", "ece_after"]].agg(["mean", "std"]).round(4))
    summ.to_csv(f"{OUT}/calibration_summary_sweep.csv")
    display(summ)
    print("\nPer-fold temperatures are reported individually: the pooled figure "
          "hides how much calibration differs between folds.")

## 9. Explanation faithfulness across **all five folds**

The reviewer's point is that a first-on-this-benchmark claim resting on one fold and 200
images is fragile. This section repeats the measurement on every fold so the numbers carry a
dispersion estimate.

In [ ]:
# Hotfix: resolve the Grad-CAM target layer structurally instead of by name.
import torch.nn as nn
import xai

def _get_target_layer(model, model_key: str = ""):
    bb = model.cnn if hasattr(model, "cnn") else getattr(model, "backbone", model)
    for attr in ("stages", "layer4", "features", "layers", "blocks"):
        if hasattr(bb, attr):
            mod = getattr(bb, attr)
            try:
                return mod[-1]
            except (TypeError, IndexError):
                return mod
    convs = [m for m in bb.modules() if isinstance(m, nn.Conv2d)]
    if convs:
        return convs[-1]
    raise ValueError(f"Could not resolve a target layer for '{model_key}' "
                     f"({type(model).__name__}).")

xai.get_target_layer = _get_target_layer
print("patched:", xai.get_target_layer.__name__)

patched: _get_target_layer


## 10. Aggregate tables and statistics

In [ ]:
# ---------- performance table, by magnification
rows = []
for mag in MAGS:
    for key in FAMILY:
        p = f"{OUT}/fold_metrics_{key}_m{mag}.csv"
        if not os.path.exists(p):
            continue
        fd = pd.read_csv(p)
        r = {"magnification": mag, "model": key,
             "params_M": models.count_parameters(
                 models.build_model(key, CFG.num_classes, False, CFG.img_size))["total_M"]}
        for m in ["img_accuracy", "img_balanced_accuracy", "img_f1_macro", "img_auc",
                  "img_mcc", "img_sensitivity", "img_specificity", "pat_accuracy"]:
            if m in fd:
                r[m] = f"{fd[m].mean():.4f} ± {fd[m].std(ddof=1):.4f}"
                r[m + "_mean"] = fd[m].mean()
        rows.append(r)
perf = pd.DataFrame(rows)
perf.to_csv(f"{OUT}/performance_by_magnification.csv", index=False)
display(perf[[c for c in perf.columns if not c.endswith("_mean")]])

# ---------- pairwise statistics within each magnification, Holm-corrected
stat_frames, tost_rows = [], []
for mag in MAGS:
    fold_scores = {}
    for key in FAMILY:
        p = f"{OUT}/fold_metrics_{key}_m{mag}.csv"
        if os.path.exists(p):
            fold_scores[key] = pd.read_csv(p)["img_balanced_accuracy"].to_numpy()
    if len(fold_scores) < 2:
        continue
    tab = S.pairwise_model_comparison(fold_scores, n_train=4, n_test=1)
    tab.insert(0, "magnification", mag)
    stat_frames.append(tab)

    # equivalence, and the effect the design could actually have detected
    for a, b in [(x, y) for i, x in enumerate(sorted(fold_scores))
                 for y in sorted(fold_scores)[i + 1:]]:
        eq = S.equivalence_test(fold_scores[a], fold_scores[b], margin=0.01)
        d = fold_scores[a] - fold_scores[b]
        k = len(d); sd = d.std(ddof=1)
        from scipy import stats as _st
        mde = (_st.t.ppf(0.975, k - 1) + _st.t.ppf(0.80, k - 1)) * sd / np.sqrt(k)
        tost_rows.append({"magnification": mag, "model_a": a, "model_b": b,
                          "mean_difference": eq["mean_difference"],
                          "margin": 0.01, "p_tost": max(eq["p_lower"], eq["p_upper"]),
                          "equivalent": eq["equivalent"],
                          "min_detectable_diff_pp": mde * 100})

if stat_frames:
    st = pd.concat(stat_frames, ignore_index=True)
    st.to_csv(f"{OUT}/stats_family_by_magnification.csv", index=False)
    display(st.round(4))
if tost_rows:
    tt = pd.DataFrame(tost_rows)
    tt.to_csv(f"{OUT}/stats_equivalence_power.csv", index=False)
    display(tt.round(4))
    print("\nRead the last column first: a difference smaller than this could not "
          "have been detected by a five-fold design at these variances, whatever "
          "the p-value says.")

,magnification,model,params_M,img_accuracy,img_balanced_accuracy,img_f1_macro,img_auc,img_mcc,img_sensitivity,img_specificity,pat_accuracy
0,40,bcaf,58.45,0.9123 ± 0.0608,0.9111 ± 0.0607,0.8997 ± 0.0620,0.9601 ± 0.0412,0.8104 ± 0.1155,0.9205 ± 0.0910,0.9017 ± 0.1031,0.9007 ± 0.0952
1,40,concat_ablation,55.35,0.9157 ± 0.0710,0.9081 ± 0.0731,0.9048 ± 0.0770,0.9652 ± 0.0317,0.8135 ± 0.1502,0.9380 ± 0.0598,0.8781 ± 0.1037,0.9507 ± 0.0521
2,40,swin_tiny,27.52,0.9086 ± 0.0528,0.9042 ± 0.0620,0.8936 ± 0.0571,0.9607 ± 0.0253,0.7963 ± 0.1098,0.9375 ± 0.0482,0.8710 ± 0.1188,0.9272 ± 0.0777
3,40,convnext_tiny,27.82,0.9223 ± 0.0564,0.9151 ± 0.0538,0.9115 ± 0.0580,0.9699 ± 0.0263,0.8332 ± 0.1077,0.9358 ± 0.0843,0.8944 ± 0.1041,0.9257 ± 0.0816
4,40,resnet50,23.51,0.8936 ± 0.0709,0.8694 ± 0.0871,0.8726 ± 0.0761,0.9150 ± 0.0694,0.7547 ± 0.1445,0.9397 ± 0.0601,0.7992 ± 0.1609,0.9257 ± 0.0525
5,100,bcaf,58.45,0.9209 ± 0.0228,0.9327 ± 0.0267,0.8972 ± 0.0362,0.9801 ± 0.0072,0.8065 ± 0.0648,0.9272 ± 0.0376,0.9381 ± 0.0539,0.9500 ± 0.0523
6,100,concat_ablation,55.35,0.9237 ± 0.0231,0.9308 ± 0.0172,0.8999 ± 0.0380,0.9674 ± 0.0179,0.8098 ± 0.0678,0.9265 ± 0.0367,0.9351 ± 0.0330,0.9625 ± 0.0342
7,100,swin_tiny,27.52,0.9255 ± 0.0309,0.9209 ± 0.0387,0.9010 ± 0.0403,0.9724 ± 0.0138,0.8089 ± 0.0787,0.9283 ± 0.0505,0.9135 ± 0.0819,0.9500 ± 0.0523
8,100,convnext_tiny,27.82,0.9312 ± 0.0321,0.9362 ± 0.0323,0.9139 ± 0.0308,0.9704 ± 0.0195,0.8337 ± 0.0583,0.9354 ± 0.0362,0.9370 ± 0.0496,0.9507 ± 0.0276
9,100,resnet50,23.51,0.8945 ± 0.0477,0.8708 ± 0.0574,0.8629 ± 0.0414,0.9402 ± 0.0312,0.7375 ± 0.0833,0.9203 ± 0.0570,0.8213 ± 0.1340,0.9132 ± 0.0563


,magnification,model_a,model_b,mean_a,mean_b,difference,cohens_d,p_ttest_corrected,p_wilcoxon,wilcoxon_at_floor,p_ttest_holm,significant
0,40,bcaf,concat_ablation,0.9111,0.9081,0.0030,0.1004,0.8882,0.8125,False,1.0000,False
1,40,bcaf,swin_tiny,0.9111,0.9042,0.0069,1.0874,0.1803,0.1250,False,1.0000,False
2,40,bcaf,convnext_tiny,0.9111,0.9151,-0.0041,-0.2877,0.6901,1.0000,False,1.0000,False
3,40,bcaf,resnet50,0.9111,0.8694,0.0416,1.0002,0.2102,0.1250,False,1.0000,False
4,40,concat_ablation,swin_tiny,0.9081,0.9042,0.0038,0.1586,0.8247,0.8125,False,1.0000,False
5,40,concat_ablation,convnext_tiny,0.9081,0.9151,-0.0071,-0.1893,0.7918,1.0000,False,1.0000,False
6,40,concat_ablation,resnet50,0.9081,0.8694,0.0386,0.9355,0.2356,0.1250,False,1.0000,False
7,40,swin_tiny,convnext_tiny,0.9042,0.9151,-0.0109,-0.6495,0.3877,0.0625,True,1.0000,False
8,40,swin_tiny,resnet50,0.9042,0.8694,0.0348,0.8820,0.2589,0.0625,True,1.0000,False
9,40,convnext_tiny,resnet50,0.9151,0.8694,0.0457,1.1753,0.1547,0.0625,True,1.0000,False


,magnification,model_a,model_b,mean_difference,margin,p_tost,equivalent,min_detectable_diff_pp
0,40,bcaf,concat_ablation,0.0030,0.01,0.3152,False,4.9904
1,40,bcaf,convnext_tiny,-0.0041,0.01,0.1998,False,2.3451
2,40,bcaf,resnet50,0.0416,0.01,0.9178,False,6.9207
3,40,bcaf,swin_tiny,0.0069,0.01,0.1631,False,1.0474
4,40,concat_ablation,convnext_tiny,-0.0071,0.01,0.4348,False,6.2126
5,40,concat_ablation,resnet50,0.0386,0.01,0.9020,False,6.8634
6,40,concat_ablation,swin_tiny,0.0038,0.01,0.2996,False,4.0199
7,40,convnext_tiny,resnet50,0.0457,0.01,0.9453,False,6.4640
8,40,convnext_tiny,swin_tiny,0.0109,0.01,0.5453,False,2.7923
9,40,resnet50,swin_tiny,-0.0348,0.01,0.8837,False,6.5571



Read the last column first: a difference smaller than this could not have been detected by a five-fold design at these variances, whatever the p-value says.


## 11. Package everything for download

In [ ]:
import zipfile
from pathlib import Path

ZIP = Path(OUT) / "bcaf_sweep_results.zip"
suffixes = (".csv", ".json", ".png", ".pdf", ".npy")
files = [f for f in Path(OUT).rglob("*")
         if f.is_file() and f.suffix.lower() in suffixes
         and f != ZIP and "checkpoints" not in f.parts and "_prior" not in f.parts]
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for f in files:
        z.write(f, f.relative_to(OUT))
print(f"{len(files)} files -> {ZIP}  ({ZIP.stat().st_size/1024/1024:.1f} MB)")

try:
    from google.colab import files as colab_files
    colab_files.download(str(ZIP))
except Exception:
    try:
        from IPython.display import FileLink, display
        display(FileLink(str(ZIP)))
    except Exception:
        print("download from:", ZIP)

print("\nSend this zip back and the manuscript tables will be regenerated from it.")

58 files -> /content/drive/MyDrive/bcaf_sweep/bcaf_sweep_results.zip  (1.1 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Send this zip back and the manuscript tables will be regenerated from it.
